# Supplementary Carry-On Shopping Answer Analysis  
## Comparison 03, chunked all-current-baseline version  
### Supplementary Check: 2026 Current Baseline vs One-Product 2023/2024 GEO-Style Rewrite

This notebook analyzes:

**2026 all-current baseline vs 2023/2024 GEO-style rewrite one-product replacement**

Important path fix: this version expects the all-current baseline to be stored as four prompt-chunk folders or files such as:

```text
baseline_1_curr
baseline_6_curr
baseline_11_curr
baseline_16_curr
```

Purpose: directly check whether replacing one 2026 current product page with its older-page GEO-style rewrite changes that product's visibility/prominence in LLM shopping answers.

Interpretation: **direct observed-current vs hypothetical-GEO comparison**.


## Expected folder structure

```text
code/
└── data/
    └── results/
        └── shopping_answers/
            ├── baseline_1_curr/       # all-current 2026 baseline answers for Q1-Q5
            ├── baseline_6_curr/       # all-current 2026 baseline answers for Q6-Q10
            ├── baseline_11_curr/      # all-current 2026 baseline answers for Q11-Q15
            ├── baseline_16_curr/      # all-current 2026 baseline answers for Q16-Q20
            └── 03_2026_current_vs_2023_2024_geo_rewrite/
                ├── away_1.txt
                ├── br_1.txt
                ├── century_1.txt
                ├── july_1.txt
                ├── rimowa_1.txt
                ├── away_6.txt
                ├── ...
                ├── away_11.txt
                ├── ...
                ├── away_16.txt
                └── ...
```

The notebook also accepts baseline chunks if they are plain text files directly under `shopping_answers`, for example:

```text
baseline_1_curr.txt
baseline_6_curr.txt
baseline_11_curr.txt
baseline_16_curr.txt
```

Each baseline chunk may internally label its five answers as `Question 1` to `Question 5`. If the folder or filename contains `_6_`, `_11_`, or `_16_`, the notebook shifts those local question IDs to the correct global question IDs.


## CSV outputs

The notebook saves CSV files to:

```text
code/data/results/tables/03_2026_current_vs_2023_2024_geo_rewrite_baseline_current_chunked_based/
```

Key saved files:

```text
file_manifest_comparison03.csv
current_baseline_parsed_answers.csv
geo_replacement_parsed_answers_by_file.csv
question_coverage_by_file.csv
question_coverage_summary.csv
geo_replacement_prompt_chunk_coverage_by_product.csv
matching_coverage_summary.csv
missing_current_baseline_matches.csv
current_baseline_question_product_metrics.csv
geo_replacement_question_product_metrics.csv
geo_vs_current_matched_question_product_deltas.csv
geo_side_product_summary_from_matched.csv
current_baseline_side_product_summary_from_matched.csv
geo_vs_current_delta_by_product.csv
overall_geo_vs_current_summary.csv
```

The most important output is:

```text
geo_vs_current_delta_by_product.csv
```

Interpretation:

```text
geo_vs_current_advantage_score > 0  => GEO rewrite stronger than 2026 current baseline
geo_vs_current_advantage_score < 0  => 2026 current baseline stronger than GEO rewrite
```


In [1]:
from pathlib import Path
import re
import json
import math
import numpy as np
import pandas as pd

try:
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.metrics.pairwise import cosine_similarity
    SKLEARN_AVAILABLE = True
except Exception:
    SKLEARN_AVAILABLE = False

pd.set_option("display.max_colwidth", 180)
pd.set_option("display.max_rows", 200)


In [2]:
# =============================
# Configuration
# =============================

PRODUCTS = ["Away", "BR", "Century", "July", "Rimowa"]
N_PRODUCTS = len(PRODUCTS)

COMPARISON_NAME = "03_2026_current_vs_2023_2024_geo_rewrite"
COMPARISON_NAME_PLURAL = "03_2026_current_vs_2023_2024_geo_rewrites"
OUTPUT_NAME = "03_2026_current_vs_2023_2024_geo_rewrite_baseline_current_chunked_based"

# Optional manual overrides.
# If automatic path detection fails, set these manually.
# For your current structure, you can leave these as None if folders are named:
# baseline_1_curr, baseline_6_curr, baseline_11_curr, baseline_16_curr.
BASELINE_CURRENT_DIR = None       # Optional single directory, if you later put all baseline chunks in one folder.
BASELINE_CURRENT_PATHS = None     # Optional list of chunk folders/files, e.g. [Path(".../baseline_1_curr"), ...]
BASELINE_CURRENT_FILES = None     # Optional direct list of .txt files.
COMPARISON_DIR = None

# Optional manual overrides for product-page text folders.
# These are used only for PAIS-style product-text-to-answer similarity.
CURRENT_TEXT_DIR = None
GEO_TEXT_DIR = None

# If False, the notebook raises an error when GEO replacement question IDs do not have matching
# all-current baseline question IDs.
ALLOW_PARTIAL_MATCH = False

# Expected final benchmark questions and prompt chunks. Used for diagnostics.
EXPECTED_QUESTION_IDS = list(range(1, 21))
EXPECTED_PROMPT_STARTS = [1, 6, 11, 16]

# If you want to restrict analysis to specific question IDs, set a list like:
# QUESTION_ID_FILTER = list(range(1, 21))
QUESTION_ID_FILTER = None


In [3]:
# =============================
# Robust path and file detection
# =============================

def get_candidate_roots():
    cwd = Path.cwd().resolve()
    roots = []
    for p in [
        cwd,
        cwd.parent,
        cwd.parent.parent,
        cwd.parent.parent.parent,
        cwd / "code",
        cwd.parent / "code",
        cwd.parent.parent / "code",
        cwd.parent.parent.parent / "code",
    ]:
        try:
            rp = p.resolve()
        except Exception:
            rp = p
        if p.exists() and rp not in roots:
            roots.append(rp)
    return roots

def find_dir(rel_candidates):
    for root in get_candidate_roots():
        for rel in rel_candidates:
            candidate = root / rel
            if candidate.exists() and candidate.is_dir():
                return candidate.resolve()
    return None

def find_all_dirs(rel_candidates):
    found = []
    for root in get_candidate_roots():
        for rel in rel_candidates:
            candidate = root / rel
            if candidate.exists() and candidate.is_dir():
                rp = candidate.resolve()
                if rp not in found:
                    found.append(rp)
    return found

def normalize_filename(name: str) -> str:
    return str(name).lower().replace("é", "e").replace(" ", "").replace("-", "_")

PRODUCT_ALIASES = {
    "Away": ["away"],
    "BR": ["br", "briggs", "riley", "briggsriley", "briggs_riley"],
    "Century": ["century", "hartmann"],
    "July": ["july"],
    "Rimowa": ["rimowa"],
}

def infer_product_from_filename(filename: str) -> str:
    name = normalize_filename(filename)
    for product, aliases in PRODUCT_ALIASES.items():
        if any(alias in name for alias in aliases):
            return product
    return "Unknown"

def is_text_like(path: Path) -> bool:
    """Accept normal text answer files, including Windows double extensions."""
    n = path.name.lower()
    return (
        n.endswith(".txt")
        or n.endswith(".txt.txt")
        or n.endswith(".text")
        or n.endswith(".text.txt")
    )

def infer_prompt_start_from_filename(path_or_filename):
    """Infer prompt chunk start from filename OR parent folder.

    Supports:
    - beis_1.txt, delsey_6.txt, monos_11.txt, tarvel_16.txt
    - baseline_1_curr/baseline.txt
    - baseline_6_curr.txt
    - baseline_curr_11.txt
    """
    raw = str(path_or_filename).lower().replace("\\", "/").replace("é", "e")
    raw_norm = raw.replace("-", "_").replace(" ", "_")

    # First, explicitly detect the all-current baseline chunk naming pattern.
    patterns = [
        r"(?:^|[/_\-])baseline[_\-]?(1|6|11|16)[_\-]?curr(?:ent)?(?:[/_.\-]|$)",
        r"(?:^|[/_\-])baseline[_\-]?curr(?:ent)?[_\-]?(1|6|11|16)(?:[/_.\-]|$)",
    ]
    for pat in patterns:
        m = re.search(pat, raw_norm)
        if m:
            return int(m.group(1))

    # Then detect simple product-answer filenames such as beis_11.txt.
    name = Path(raw_norm).name
    m = re.search(r"(?:^|[_\-])(1|6|11|16)(?=(?:\.[a-z0-9]+)+$|$)", name)
    if m:
        return int(m.group(1))

    # Final fallback: token scan, but only for the expected prompt starts.
    tokens = re.split(r"[^a-z0-9]+", raw_norm)
    for tok in ["1", "6", "11", "16"]:
        if tok in tokens:
            return int(tok)
    return None

def infer_prompt_set_from_question_ids(question_ids):
    if not question_ids:
        return np.nan, np.nan, ""
    qmin, qmax = int(min(question_ids)), int(max(question_ids))
    return qmin, qmax, f"{qmin}-{qmax}"

def sort_prompt_set_labels(labels):
    """Sort labels like 1-5, 6-10, 11-15, 16-20 in numeric order."""
    clean = [str(x) for x in labels if str(x).strip() and str(x).lower() != "nan"]
    def key(label):
        m = re.search(r"(\d+)", label)
        return int(m.group(1)) if m else 10**9
    return sorted(set(clean), key=key)

def join_prompt_set_labels(labels):
    return ";".join(sort_prompt_set_labels(labels))

def discover_text_files_under_path(path: Path):
    """Return text-like files under a directory, or the file itself if text-like."""
    path = Path(path)
    if path.is_file() and is_text_like(path):
        return [path.resolve()]
    if not path.exists() or not path.is_dir():
        return []
    candidates = []
    for pattern in ["*.txt*", "*.text*"]:
        candidates.extend([p for p in path.rglob(pattern) if p.is_file() and is_text_like(p)])
    return sorted(set(p.resolve() for p in candidates))

def is_baseline_current_chunk_name(name: str) -> bool:
    n = normalize_filename(name)
    compact = re.sub(r"[^a-z0-9]", "", n)
    allowed = []
    for start in EXPECTED_PROMPT_STARTS:
        allowed.extend([
            f"baseline{start}curr",
            f"baseline{start}current",
            f"baselinecurr{start}",
            f"baselinecurrent{start}",
        ])
    return any(token in compact for token in allowed)

def discover_shopping_answer_roots():
    roots = []
    rels = [
        Path("data/results/shopping_answers"),
        Path("code/data/results/shopping_answers"),
        Path("results/shopping_answers"),
    ]
    for root in get_candidate_roots():
        for rel in rels:
            candidate = (root / rel).resolve()
            if candidate.exists() and candidate.is_dir() and candidate not in roots:
                roots.append(candidate)
    return roots

def discover_baseline_current_chunk_paths():
    """Find baseline_1_curr, baseline_6_curr, baseline_11_curr, baseline_16_curr folders/files."""
    paths = []
    for shopping_root in discover_shopping_answer_roots():
        for child in shopping_root.iterdir():
            if is_baseline_current_chunk_name(child.name):
                rp = child.resolve()
                if rp not in paths:
                    paths.append(rp)
    # Sort by prompt start so the manifest is easy to read.
    return sorted(paths, key=lambda p: (infer_prompt_start_from_filename(str(p)) or 10**9, str(p)))

def discover_baseline_current_files(baseline_current_dir: Path = None, baseline_current_paths=None):
    """Find all all-current 2026 baseline answer files.

    Supports:
    1. A single folder containing all current-baseline files.
    2. Four chunk folders/files named baseline_1_curr, baseline_6_curr, baseline_11_curr, baseline_16_curr.
    3. Manual BASELINE_CURRENT_FILES.
    """
    discovered_paths = []

    if baseline_current_paths is not None:
        for p in baseline_current_paths:
            pp = Path(p).resolve()
            if pp.exists() and pp not in discovered_paths:
                discovered_paths.append(pp)

    if baseline_current_dir is not None:
        pp = Path(baseline_current_dir).resolve()
        if pp.exists() and pp not in discovered_paths:
            discovered_paths.append(pp)

    # If no manual/single-folder path is provided, automatically find chunked baselines.
    if not discovered_paths:
        discovered_paths.extend(discover_baseline_current_chunk_paths())

    candidates = []
    for p in discovered_paths:
        candidates.extend(discover_text_files_under_path(p))

    files = sorted(set(candidates), key=lambda p: (infer_prompt_start_from_filename(str(p)) or 10**9, str(p)))

    valid = []
    for p in files:
        n = normalize_filename(str(p))
        compact = re.sub(r"[^a-z0-9]", "", n)
        banned_tokens = [
            "vsgeo", "georewrite", "geo_rewrite", "oneproduct",
            "20232024vsgeo", "2026vsgeo"
        ]
        if any(tok in compact for tok in banned_tokens):
            continue
        valid.append(p)
    return valid, discovered_paths

def is_comparison03_file(filename: str) -> bool:
    """Accept one-product GEO-rewrite replacement answer files for Comparison 03.

    In your folder, files can be simple names such as beis_1.txt, delsey_6.txt,
    monos_11.txt, or typo variants such as tarvel_1.txt. The notebook only requires
    a recognizable product name and a text-like extension.
    """
    if filename.startswith("."):
        return False
    if not is_text_like(Path(filename)):
        return False
    if infer_product_from_filename(filename) == "Unknown":
        return False
    compact = re.sub(r"[^a-z0-9]", "", normalize_filename(filename))
    if "baseline" in compact or "allcurrent" in compact or "currentbaseline" in compact:
        return False
    return True

if BASELINE_CURRENT_DIR is None:
    # Optional fallback: support older single-folder names too, but chunked baseline_*_curr is preferred.
    BASELINE_CURRENT_DIR = find_dir([
        Path("data/results/shopping_answers/baseline_curr"),
        Path("code/data/results/shopping_answers/baseline_curr"),
        Path("results/shopping_answers/baseline_curr"),
        Path("data/results/shopping_answers/baseline_current"),
        Path("code/data/results/shopping_answers/baseline_current"),
        Path("results/shopping_answers/baseline_current"),
        Path("data/results/shopping_answers/current_baseline"),
        Path("code/data/results/shopping_answers/current_baseline"),
        Path("results/shopping_answers/current_baseline"),
    ])
else:
    BASELINE_CURRENT_DIR = Path(BASELINE_CURRENT_DIR).resolve()

if BASELINE_CURRENT_PATHS is not None:
    BASELINE_CURRENT_PATHS = [Path(p).resolve() for p in BASELINE_CURRENT_PATHS]

if COMPARISON_DIR is None:
    COMPARISON_DIR = find_dir([
        Path("data/results/shopping_answers") / COMPARISON_NAME,
        Path("data/results/shopping_answers") / COMPARISON_NAME_PLURAL,
        Path("code/data/results/shopping_answers") / COMPARISON_NAME,
        Path("code/data/results/shopping_answers") / COMPARISON_NAME_PLURAL,
        Path("results/shopping_answers") / COMPARISON_NAME,
        Path("results/shopping_answers") / COMPARISON_NAME_PLURAL,
    ])
else:
    COMPARISON_DIR = Path(COMPARISON_DIR).resolve()

if CURRENT_TEXT_DIR is None:
    CURRENT_TEXT_DIR = find_dir([
        Path("data/filtered/current"),
        Path("code/data/filtered/current"),
        Path("data/results/filtered/current"),
        Path("code/data/results/filtered/current"),
        Path("data/raw_2026"),
        Path("code/data/raw_2026"),
    ])
else:
    CURRENT_TEXT_DIR = Path(CURRENT_TEXT_DIR).resolve()

if GEO_TEXT_DIR is None:
    GEO_TEXT_DIR = find_dir([
        Path("data/geo_rewrites"),
        Path("code/data/geo_rewrites"),
        Path("data/filtered/geo_rewrites"),
        Path("code/data/filtered/geo_rewrites"),
        Path("data/results/geo_rewrites"),
        Path("code/data/results/geo_rewrites"),
    ])
else:
    GEO_TEXT_DIR = Path(GEO_TEXT_DIR).resolve()

if BASELINE_CURRENT_FILES is None:
    BASELINE_CURRENT_FILES, BASELINE_CURRENT_DISCOVERED_PATHS = discover_baseline_current_files(
        BASELINE_CURRENT_DIR,
        BASELINE_CURRENT_PATHS,
    )
else:
    BASELINE_CURRENT_FILES = [Path(p).resolve() for p in BASELINE_CURRENT_FILES]
    BASELINE_CURRENT_DISCOVERED_PATHS = []

if COMPARISON_DIR is not None:
    candidates = []
    for pattern in ["*.txt*", "*.text*"]:
        candidates.extend([p for p in COMPARISON_DIR.glob(pattern) if p.is_file() and is_text_like(p)])
    all_comparison_txt_files = sorted(set(candidates))
else:
    all_comparison_txt_files = []

TXT_FILES = [p for p in all_comparison_txt_files if is_comparison03_file(p.name)]
IGNORED_TXT_FILES = [p for p in all_comparison_txt_files if p not in TXT_FILES]

print("Current working directory:", Path.cwd().resolve())
print("Detected single all-current baseline directory fallback:", BASELINE_CURRENT_DIR)
print("Detected all-current baseline chunk paths:")
for p in BASELINE_CURRENT_DISCOVERED_PATHS:
    print("  -", p, "prompt_start=", infer_prompt_start_from_filename(str(p)))
print("Detected all-current baseline files:")
for p in BASELINE_CURRENT_FILES:
    print("  -", p, "prompt_start=", infer_prompt_start_from_filename(str(p)))
print("Detected comparison directory:", COMPARISON_DIR)
print("Detected current 2026 text directory:", CURRENT_TEXT_DIR)
print("Detected GEO rewrite text directory:", GEO_TEXT_DIR)

if not BASELINE_CURRENT_FILES:
    raise FileNotFoundError(
        "Could not find all-current 2026 baseline .txt/.text files. "
        "Expected folders/files like baseline_1_curr, baseline_6_curr, baseline_11_curr, baseline_16_curr. "
        "If needed, set BASELINE_CURRENT_PATHS or BASELINE_CURRENT_FILES manually in the configuration cell."
    )

if COMPARISON_DIR is None:
    raise FileNotFoundError(
        "Could not find the Comparison 03 folder. "
        "Please set COMPARISON_DIR manually in the configuration cell."
    )

print(f"\nIncluded {len(TXT_FILES)} one-product-GEO-rewrite replacement txt-like files:")
for p in TXT_FILES:
    print("  -", p.name, "=>", infer_product_from_filename(p.name), "prompt_start=", infer_prompt_start_from_filename(str(p)))

print(f"\nIgnored {len(IGNORED_TXT_FILES)} txt-like files:")
for p in IGNORED_TXT_FILES:
    reason = "not recognized as one-product GEO replacement file"
    if infer_product_from_filename(p.name) == "Unknown":
        reason = "unknown product"
    print("  -", p.name, f"({reason})")

if not TXT_FILES:
    raise FileNotFoundError(f"No one-product-GEO replacement .txt/.text files found in {COMPARISON_DIR}")


Current working directory: C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\supplementary_check\code
Detected single all-current baseline directory fallback: C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\supplementary_check\results\shopping_answers\baseline_curr
Detected all-current baseline chunk paths:
  - C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\supplementary_check\results\shopping_answers\baseline_curr prompt_start= None
Detected all-current baseline files:
  - C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\supplementary_check\results\shopping_answers\baseline_curr\baseline_1.txt prompt_start= 1
  - C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\supplementary_check\results\shopping_answers\baseline_curr\baseline_6.txt prompt_start= 6
  - C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\supplementary_check\results\shopping_answers\baseline_curr\baseline_11.txt prompt_start= 11
  

In [4]:
# =============================
# Output directory and file manifest
# =============================

# Based on your structure, if COMPARISON_DIR is:
# code/data/results/shopping_answers/03_...
# then tables go to:
# code/data/results/tables/03_..._baseline_current_chunked_based
RESULTS_DIR = COMPARISON_DIR.parents[1]  # .../results
OUTPUT_DIR = RESULTS_DIR / "tables" / OUTPUT_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

file_manifest_rows = []
for p in BASELINE_CURRENT_FILES:
    file_manifest_rows.append({
        "file_role": "baseline_current_included",
        "path": str(p),
        "filename": p.name,
        "detected_product": "",
        "prompt_start_from_filename": infer_prompt_start_from_filename(str(p)),
        "included": True,
        "reason": "all-current 2026 baseline answer file from baseline_*_curr chunk",
    })
for p in TXT_FILES:
    file_manifest_rows.append({
        "file_role": "comparison03_geo_rewrite_replacement_included",
        "path": str(p),
        "filename": p.name,
        "detected_product": infer_product_from_filename(p.name),
        "prompt_start_from_filename": infer_prompt_start_from_filename(str(p)),
        "included": True,
        "reason": "one-product GEO rewrite replacement file against all-current context",
    })
for p in IGNORED_TXT_FILES:
    file_manifest_rows.append({
        "file_role": "comparison03_ignored",
        "path": str(p),
        "filename": p.name,
        "detected_product": infer_product_from_filename(p.name),
        "prompt_start_from_filename": infer_prompt_start_from_filename(str(p)),
        "included": False,
        "reason": "not recognized as one-product GEO replacement file",
    })

file_manifest_comparison03 = pd.DataFrame(file_manifest_rows)

print("Results directory:", RESULTS_DIR)
print("CSV output directory:", OUTPUT_DIR)
display(file_manifest_comparison03)


Results directory: C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\supplementary_check\results
CSV output directory: C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\supplementary_check\results\tables\03_2026_current_vs_2023_2024_geo_rewrite_baseline_current_chunked_based


,file_role,path,filename,detected_product,prompt_start_from_filename,included,reason
0,baseline_current_included,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\supplementary_check\results\shopping_answers\baseline_curr\baseline_1.txt,baseline_1.txt,,1,True,all-current 2026 baseline answer file from baseline_*_curr chunk
1,baseline_current_included,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\supplementary_check\results\shopping_answers\baseline_curr\baseline_6.txt,baseline_6.txt,,6,True,all-current 2026 baseline answer file from baseline_*_curr chunk
2,baseline_current_included,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\supplementary_check\results\shopping_answers\baseline_curr\baseline_11.txt,baseline_11.txt,,11,True,all-current 2026 baseline answer file from baseline_*_curr chunk
3,baseline_current_included,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\supplementary_check\results\shopping_answers\baseline_curr\baseline_16.txt,baseline_16.txt,,16,True,all-current 2026 baseline answer file from baseline_*_curr chunk
4,comparison03_geo_rewrite_replacement_included,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\supplementary_check\results\shopping_answers\03_2026_current_vs_2023_2024_geo_rewrite\away_1.txt,away_1.txt,Away,1,True,one-product GEO rewrite replacement file against all-current context
5,comparison03_geo_rewrite_replacement_included,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\supplementary_check\results\shopping_answers\03_2026_current_vs_2023_2024_geo_rewrite\away_11.txt,away_11.txt,Away,11,True,one-product GEO rewrite replacement file against all-current context
6,comparison03_geo_rewrite_replacement_included,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\supplementary_check\results\shopping_answers\03_2026_current_vs_2023_2024_geo_rewrite\away_16.txt,away_16.txt,Away,16,True,one-product GEO rewrite replacement file against all-current context
7,comparison03_geo_rewrite_replacement_included,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\supplementary_check\results\shopping_answers\03_2026_current_vs_2023_2024_geo_rewrite\away_6.txt,away_6.txt,Away,6,True,one-product GEO rewrite replacement file against all-current context
8,comparison03_geo_rewrite_replacement_included,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\supplementary_check\results\shopping_answers\03_2026_current_vs_2023_2024_geo_rewrite\br_1.txt,br_1.txt,BR,1,True,one-product GEO rewrite replacement file against all-current context
9,comparison03_geo_rewrite_replacement_included,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\supplementary_check\results\shopping_answers\03_2026_current_vs_2023_2024_geo_rewrite\br_11.txt,br_11.txt,BR,11,True,one-product GEO rewrite replacement file against all-current context


## Parse answer files

The parser expects question blocks such as:

```text
Question 1:
Answer:
...

Question 2:
Answer:
...

Overall shopper takeaway:
...
```

Safety fix: if a file named `_6`, `_11`, or `_16` internally labels the five local blocks as `Question 1` to `Question 5`, the notebook shifts them to the correct global question IDs.


In [5]:
# =============================
# Parsing functions
# =============================

QUESTION_BLOCK_RE = re.compile(
    r"Question\s*(\d+)\s*:\s*(.*?)(?=\n\s*Question\s*\d+\s*:|\n\s*Overall shopper takeaway\s*:|\Z)",
    flags=re.IGNORECASE | re.DOTALL,
)

OVERALL_RE = re.compile(
    r"Overall shopper takeaway\s*:\s*(.*)\Z",
    flags=re.IGNORECASE | re.DOTALL,
)

def clean_answer_block(block: str) -> str:
    block = block.strip()
    block = re.sub(r"^\s*Answer\s*:\s*", "", block, flags=re.IGNORECASE).strip()
    block = re.sub(r"\n{3,}", "\n\n", block)
    return block

def maybe_shift_question_id(raw_qid: int, raw_qids_all, filename_prompt_start):
    """Normalize local Q1-Q5 labels using the filename chunk suffix when needed."""
    if filename_prompt_start in {6, 11, 16}:
        # Some LLM outputs label each 5-question chunk locally as Question 1-5.
        # If all parsed IDs are in 1-5, map them to the global question range.
        if raw_qids_all and min(raw_qids_all) >= 1 and max(raw_qids_all) <= 5:
            return filename_prompt_start + raw_qid - 1, True
    return raw_qid, False

def parse_answer_file(path: Path, run_type: str, focal_geo_product: str = None) -> pd.DataFrame:
    text = path.read_text(encoding="utf-8-sig", errors="replace").replace("\r\n", "\n").replace("\r", "\n")
    matches = list(QUESTION_BLOCK_RE.finditer(text))
    raw_qids_all = [int(m.group(1)) for m in matches]
    filename_prompt_start = infer_prompt_start_from_filename(str(path))

    rows = []
    normalized_qids_found = []

    for m in matches:
        raw_qid = int(m.group(1))
        qid, shifted = maybe_shift_question_id(raw_qid, raw_qids_all, filename_prompt_start)
        if QUESTION_ID_FILTER is not None and qid not in set(QUESTION_ID_FILTER):
            continue
        answer = clean_answer_block(m.group(2))
        normalized_qids_found.append(qid)
        rows.append({
            "source_file": f"{path.parent.name}/{path.name}",
            "source_path": str(path),
            "run_type": run_type,
            "focal_geo_product": focal_geo_product,
            "comparison": COMPARISON_NAME,
            "filename_prompt_start": filename_prompt_start,
            "raw_question_id": raw_qid,
            "question_id_was_shifted_from_filename": bool(shifted),
            "question_id": qid,
            "answer_text": answer,
        })

    overall = ""
    om = OVERALL_RE.search(text)
    if om:
        overall = om.group(1).strip()

    qmin, qmax, qlabel = infer_prompt_set_from_question_ids(normalized_qids_found)

    df = pd.DataFrame(rows)
    if not df.empty:
        df["prompt_set_start"] = qmin
        df["prompt_set_end"] = qmax
        df["prompt_set_label"] = qlabel
        df["overall_shopper_takeaway"] = overall
    return df

baseline_parts = [
    parse_answer_file(p, run_type="baseline_all_current_2026", focal_geo_product=None)
    for p in BASELINE_CURRENT_FILES
]
baseline_parts = [df for df in baseline_parts if not df.empty]
baseline_answers = pd.concat(baseline_parts, ignore_index=True) if baseline_parts else pd.DataFrame()

geo_parts = [
    parse_answer_file(
        p,
        run_type="one_product_geo_rewrite_against_current",
        focal_geo_product=infer_product_from_filename(p.name),
    )
    for p in TXT_FILES
]
geo_parts = [df for df in geo_parts if not df.empty]
geo_answers = pd.concat(geo_parts, ignore_index=True) if geo_parts else pd.DataFrame()

if baseline_answers.empty:
    raise ValueError("No question blocks were parsed from all-current baseline files.")

if geo_answers.empty:
    raise ValueError("No question blocks were parsed from Comparison 03 GEO-replacement files.")

# Remove exact duplicate parsed rows if the same file was accidentally included twice.
dedup_cols = ["source_file", "run_type", "focal_geo_product", "question_id", "answer_text"]
baseline_answers = baseline_answers.drop_duplicates(subset=dedup_cols).reset_index(drop=True)
geo_answers = geo_answers.drop_duplicates(subset=dedup_cols).reset_index(drop=True)

baseline_qids = sorted(map(int, baseline_answers["question_id"].unique()))
geo_qids = sorted(map(int, geo_answers["question_id"].unique()))

print("All-current baseline parsed rows:", len(baseline_answers))
display(baseline_answers[["source_file", "filename_prompt_start", "raw_question_id", "question_id", "question_id_was_shifted_from_filename", "prompt_set_label", "answer_text"]].head(30))

print("GEO-replacement parsed rows:", len(geo_answers))
display(geo_answers[["source_file", "focal_geo_product", "filename_prompt_start", "raw_question_id", "question_id", "question_id_was_shifted_from_filename", "prompt_set_label", "answer_text"]].head(30))

print("Focal GEO replacement products detected by file:")
display(
    geo_answers[["source_file", "focal_geo_product", "prompt_set_label"]]
    .drop_duplicates()
    .sort_values(["focal_geo_product", "prompt_set_label", "source_file"])
)

coverage_by_file = pd.concat([
    baseline_answers.groupby(["run_type", "source_file"], as_index=False).agg(
        focal_geo_product=("focal_geo_product", lambda x: ""),
        question_ids=("question_id", lambda x: ",".join(map(str, sorted(set(map(int, x)))))),
        n_questions=("question_id", "nunique"),
        prompt_set_label=("prompt_set_label", lambda x: join_prompt_set_labels(x)),
        shifted_any=("question_id_was_shifted_from_filename", "max"),
    ),
    geo_answers.groupby(["run_type", "source_file", "focal_geo_product"], as_index=False).agg(
        question_ids=("question_id", lambda x: ",".join(map(str, sorted(set(map(int, x)))))),
        n_questions=("question_id", "nunique"),
        prompt_set_label=("prompt_set_label", lambda x: join_prompt_set_labels(x)),
        shifted_any=("question_id_was_shifted_from_filename", "max"),
    )
], ignore_index=True, sort=False)

print("Question coverage by parsed file:")
display(coverage_by_file.sort_values(["run_type", "focal_geo_product", "source_file"]))

question_coverage_summary = pd.DataFrame([{
    "baseline_current_question_ids": ",".join(map(str, baseline_qids)),
    "geo_replacement_question_ids": ",".join(map(str, geo_qids)),
    "expected_question_ids": ",".join(map(str, EXPECTED_QUESTION_IDS)),
    "missing_from_baseline_current_vs_expected": ",".join(map(str, sorted(set(EXPECTED_QUESTION_IDS) - set(baseline_qids)))),
    "missing_from_geo_replacement_vs_expected": ",".join(map(str, sorted(set(EXPECTED_QUESTION_IDS) - set(geo_qids)))),
}])

print("Overall question coverage summary:")
display(question_coverage_summary)

geo_prompt_chunk_coverage_by_product = []
for product in PRODUCTS:
    g = geo_answers[geo_answers["focal_geo_product"] == product]
    starts = sorted(set(int(x) for x in g["prompt_set_start"].dropna().unique()))
    qids = sorted(set(int(x) for x in g["question_id"].dropna().unique()))
    geo_prompt_chunk_coverage_by_product.append({
        "product": product,
        "n_geo_replacement_files_detected": g["source_file"].nunique(),
        "detected_prompt_starts": ",".join(map(str, starts)),
        "missing_prompt_starts_vs_expected": ",".join(map(str, sorted(set(EXPECTED_PROMPT_STARTS) - set(starts)))),
        "n_unique_questions": len(qids),
        "question_ids": ",".join(map(str, qids)),
        "complete_q1_to_q20": set(qids) == set(EXPECTED_QUESTION_IDS),
    })
geo_prompt_chunk_coverage_by_product = pd.DataFrame(geo_prompt_chunk_coverage_by_product)

print("GEO replacement prompt chunk coverage by product:")
display(geo_prompt_chunk_coverage_by_product)


All-current baseline parsed rows: 20


,source_file,filename_prompt_start,raw_question_id,question_id,question_id_was_shifted_from_filename,prompt_set_label,answer_text
0,baseline_curr/baseline_1.txt,1,1,1,False,1-5,"For a 3-5 day trip, I would recommend the Away Carry-On because its product page directly says it is generally best for 3-5 day trips and sized for about 5-7 full outfits. [Awa..."
1,baseline_curr/baseline_1.txt,1,2,2,False,1-5,"The Briggs & Riley Baseline Essential 22"" Carry-On Expandable Spinner seems strongest if durability and long-term protection matter most because it is made from premium ultra-d..."
2,baseline_curr/baseline_1.txt,1,3,3,False,1-5,"For easiest movement through airports, streets, and crowded spaces, July looks like the best all-around choice because it has SilentMove 360° spinner wheels described as extrem..."
3,baseline_curr/baseline_1.txt,1,4,4,False,1-5,"Briggs & Riley seems best for packing organization because it combines CX® Smart Packing Technology that expands up to 30% and compresses back to carry-on size, an exterior Out..."
4,baseline_curr/baseline_1.txt,1,5,5,False,1-5,"Away gives the best overall purchase confidence for most shoppers because its page clearly states the 21.7"" x 14.4"" x 9"" exterior size, 7.5 lb weight, 41 L capacity, fit in ove..."
5,baseline_curr/baseline_6.txt,6,6,6,False,6-10,"For a stylish overpacker on a 3-5 day trip, I would recommend the Hartmann Century Deluxe Carry-On if the priority is a polished look plus a little extra room: it is described ..."
6,baseline_curr/baseline_6.txt,6,7,7,False,6-10,"The Briggs & Riley Baseline Essential 22-inch Carry-On Expandable Spinner seems best for a shopper who wants a traditional luggage-brand feel, strong protection features, smoot..."
7,baseline_curr/baseline_6.txt,6,8,8,False,6-10,The product-page information does not identify one suitcase that is both definitively the lightest and the cheapest hard-shell option. RIMOWA calls the Essential Lite the light...
8,baseline_curr/baseline_6.txt,6,9,9,False,6-10,"July stands out among the hard-shell options for protection details because it combines an aerospace-grade German polycarbonate shell, reinforced bumpers, an integrated TSA loc..."
9,baseline_curr/baseline_6.txt,6,10,10,False,6-10,"For business or tech-heavy travel, Briggs & Riley’s Baseline Essential 22-inch Carry-On Expandable Spinner seems the most useful overall because it combines a SpeedThru pocket ..."


GEO-replacement parsed rows: 95


,source_file,focal_geo_product,filename_prompt_start,raw_question_id,question_id,question_id_was_shifted_from_filename,prompt_set_label,answer_text
0,03_2026_current_vs_2023_2024_geo_rewrite/away_1.txt,Away,1,1,1,False,1-5,"I would recommend Away for a 3-5 day trip because its page specifically says The Carry-On is made for quick trips of 3-5 days, fits 5-7 outfits, and is sized to fit in all over..."
1,03_2026_current_vs_2023_2024_geo_rewrite/away_1.txt,Away,1,2,2,False,1-5,"For durability and protecting belongings, I would put July and Briggs & Riley at the top, with July best for hard-shell impact protection and Briggs & Riley best for long-term ..."
2,03_2026_current_vs_2023_2024_geo_rewrite/away_1.txt,Away,1,3,3,False,1-5,"For easiest movement through airports, streets, and crowded spaces, I would recommend Away or July depending on the surface you expect most. [Away] [July] Away is especially st..."
3,03_2026_current_vs_2023_2024_geo_rewrite/away_1.txt,Away,1,4,4,False,1-5,"For packing organization, I would recommend Briggs & Riley as the most complete organizer. [Briggs & Riley] It offers CX® Smart Packing Technology that expands up to 30% and co..."
4,03_2026_current_vs_2023_2024_geo_rewrite/away_1.txt,Away,1,5,5,False,1-5,"For best overall purchase confidence across size, lock/security, warranty or trial, and travel limitations, I would choose Away as the safest all-around recommendation. [Away] ..."
5,03_2026_current_vs_2023_2024_geo_rewrite/away_11.txt,Away,11,11,11,False,11-15,"The safest option for stricter carry-on rules appears to be Away because the page says it is the brand’s smallest suitcase, is “sized to fit in all overhead bins,” “meets carry..."
6,03_2026_current_vs_2023_2024_geo_rewrite/away_11.txt,Away,11,12,12,False,11-15,"July gives the strongest easy-clean and good-looking-after-travel evidence because it lists a water-resistant and stain-proof nylon lining, says all lining is stain-resistant a..."
7,03_2026_current_vs_2023_2024_geo_rewrite/away_11.txt,Away,11,13,13,False,11-15,Away provides the most concrete sustainability-related information because it says its Classic suitcases ship in exterior packaging that is 100% recyclable and that the drawstr...
8,03_2026_current_vs_2023_2024_geo_rewrite/away_11.txt,Away,11,14,14,False,11-15,"For the broadest travel-day convenience package, Briggs & Riley seems the most useful because it combines expansion-and-compression packing, a garment carrier, flat-packing Out..."
9,03_2026_current_vs_2023_2024_geo_rewrite/away_11.txt,Away,11,15,15,False,11-15,"Away has one of the strongest information-quality pages because it provides airline-fit language, exterior and interior measurements, weight, capacity, wheel-included measureme..."


Focal GEO replacement products detected by file:


...

,run_type,source_file,focal_geo_product,question_ids,n_questions,prompt_set_label,shifted_any
0,baseline_all_current_2026,baseline_curr/baseline_1.txt,,"1,2,3,4,5",5,1-5,False
1,baseline_all_current_2026,baseline_curr/baseline_11.txt,,"11,12,13,14,15",5,11-15,False
2,baseline_all_current_2026,baseline_curr/baseline_16.txt,,"16,17,18,19,20",5,16-20,False
3,baseline_all_current_2026,baseline_curr/baseline_6.txt,,"6,7,8,9,10",5,6-10,False
4,one_product_geo_rewrite_against_current,03_2026_current_vs_2023_2024_geo_rewrite/away_1.txt,Away,"1,2,3,4,5",5,1-5,False
5,one_product_geo_rewrite_against_current,03_2026_current_vs_2023_2024_geo_rewrite/away_11.txt,Away,"11,12,13,14,15",5,11-15,False
6,one_product_geo_rewrite_against_current,03_2026_current_vs_2023_2024_geo_rewrite/away_16.txt,Away,"16,17,18,19,20",5,16-20,False
7,one_product_geo_rewrite_against_current,03_2026_current_vs_2023_2024_geo_rewrite/away_6.txt,Away,"6,7,8,9,10",5,6-10,False
8,one_product_geo_rewrite_against_current,03_2026_current_vs_2023_2024_geo_rewrite/br_1.txt,BR,"1,2,3,4,5",5,1-5,False
9,one_product_geo_rewrite_against_current,03_2026_current_vs_2023_2024_geo_rewrite/br_11.txt,BR,"11,12,13,14,15",5,11-15,False


Overall question coverage summary:


,baseline_current_question_ids,geo_replacement_question_ids,expected_question_ids,missing_from_baseline_current_vs_expected,missing_from_geo_replacement_vs_expected
0,"1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20","1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20","1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20",,


GEO replacement prompt chunk coverage by product:


,product,n_geo_replacement_files_detected,detected_prompt_starts,missing_prompt_starts_vs_expected,n_unique_questions,question_ids,complete_q1_to_q20
0,Away,4,"1,6,11,16",,20,"1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20",True
1,BR,4,"1,6,11,16",,20,"1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20",True
2,Century,4,"1,6,11,16",,20,"1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20",True
3,July,4,"1,6,11,16",,20,"1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20",True
4,Rimowa,3,"6,11,16",1,15,"6,7,8,9,10,11,12,13,14,15,16,17,18,19,20",False


## Load product-page texts

The PAIS-style score uses product-page text similarity to cited answer sentences.

For Comparison 03, the baseline side uses `current_2026` product text. The replacement side uses `geo_rewrite_2023_2024` text only for the focal product and `current_2026` text for the other products.

If the text files are not found, the notebook still computes citation/rank/feature metrics, while the TF-IDF and n-gram parts of PAIS are set to zero.


In [6]:
# =============================
# Product text loading
# =============================

PREFERRED_CURRENT_TEXT_FILENAMES = {
    "BR": [
        "beis_carry_on_roller_filtered.txt",
        "beis_the_carry_on_roller_filtered.txt",
        "beis_the_carry_on_roller.txt",
        "beis_carry_on_roller.txt",
    ],
    "Century": [
        "delsey_chatelet_air_carry_on_plus_filtered.txt",
        "delsey_chatelet_air_2_0_carry_on_plus_filtered.txt",
        "delsey_chatelet_air_carry_on_plus.txt",
        "delsey_chatelet_air_2_0_carry_on_plus.txt",
    ],
    "July": [
        "inusa_ally_carry_on_filtered.txt",
        "inusa_ally_hardside_20_inch_carry_on_filtered.txt",
        "inusa_ally_carry_on.txt",
    ],
    "Away": [
        "monos_carry_on_filtered.txt",
        "monos_carry_on.txt",
    ],
    "Rimowa": [
        "travelpro_platinum_elite_carry_on_filtered.txt",
        "travelpro_platinum_elite_hardside_carry_on_filtered.txt",
        "travelpro_platinum_elite_carry_on.txt",
    ],
}

PREFERRED_GEO_TEXT_FILENAMES = {
    "BR": [
        "beis_the_carry_on_roller_in_beige.txt",
        "beis_carry_on_roller_geo_rewrite.txt",
        "beis_the_carry_on_roller.txt",
    ],
    "Century": [
        "delsey_chatelet_air_carry_on_plus.txt",
        "delsey_chatelet_air_2_0_carry_on_plus.txt",
        "delsey_chatelet_air_carry_on_plus_geo_rewrite.txt",
    ],
    "July": [
        "inusa_ally_carry_on.txt",
        "inusa_ally_hardside_20_inch_carry_on_geo_rewrite.txt",
    ],
    "Away": [
        "monos_carry_on.txt",
        "monos_carry_on_geo_rewrite.txt",
    ],
    "Rimowa": [
        "travelpro_platinum_elite_carry_on.txt",
        "travelpro_platinum_elite_hardside_carry_on_geo_rewrite.txt",
    ],
}

def read_text_if_exists(path):
    if path is not None and path.exists():
        return path.read_text(encoding="utf-8-sig", errors="replace")
    return ""

def find_product_text_file(folder: Path, product: str, preferred_filenames):
    """Find product text robustly.

    First tries known preferred filenames, then falls back to any text-like file
    containing one of the product aliases. This helps when local file names differ.
    """
    if folder is None or not Path(folder).exists():
        return None

    folder = Path(folder)

    for filename in preferred_filenames.get(product, []):
        direct = folder / filename
        if direct.exists() and direct.is_file():
            return direct.resolve()

    # Try recursive exact basename match.
    for filename in preferred_filenames.get(product, []):
        hits = [p for p in folder.rglob(filename) if p.is_file()]
        if hits:
            return sorted(hits, key=lambda p: len(str(p)))[0].resolve()

    aliases = PRODUCT_ALIASES.get(product, [])
    candidates = []
    for pattern in ["*.txt*", "*.text*"]:
        candidates.extend([p for p in folder.rglob(pattern) if p.is_file() and is_text_like(p)])
    candidates = sorted(set(candidates), key=lambda p: (len(str(p)), str(p).lower()))

    for p in candidates:
        name = normalize_filename(p.name)
        if any(alias in name for alias in aliases):
            return p.resolve()

    return None

product_texts = {}
load_rows = []

for version_status, folder, preferred in [
    ("current_2026", CURRENT_TEXT_DIR, PREFERRED_CURRENT_TEXT_FILENAMES),
    ("geo_rewrite_2023_2024", GEO_TEXT_DIR, PREFERRED_GEO_TEXT_FILENAMES),
]:
    for product in PRODUCTS:
        path = find_product_text_file(folder, product, preferred)
        text = read_text_if_exists(path)
        product_texts[(version_status, product)] = text
        load_rows.append({
            "version_status": version_status,
            "product": product,
            "path": str(path) if path is not None else "",
            "loaded": bool(text.strip()),
            "characters": len(text),
        })

product_text_load_status = pd.DataFrame(load_rows)
display(product_text_load_status)


,version_status,product,path,loaded,characters
0,current_2026,Away,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\supplementary_check\data\filtered\current\away.txt,True,6702
1,current_2026,BR,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\supplementary_check\data\filtered\current\br.txt,True,4191
2,current_2026,Century,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\supplementary_check\data\filtered\current\century.txt,True,1987
3,current_2026,July,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\supplementary_check\data\filtered\current\july.txt,True,3029
4,current_2026,Rimowa,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\supplementary_check\data\filtered\current\rimowa.txt,True,1727
5,geo_rewrite_2023_2024,Away,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\supplementary_check\data\geo_rewrites\away.txt,True,4286
6,geo_rewrite_2023_2024,BR,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\supplementary_check\data\geo_rewrites\br.txt,True,4499
7,geo_rewrite_2023_2024,Century,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\supplementary_check\data\geo_rewrites\century.txt,True,1604
8,geo_rewrite_2023_2024,July,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\supplementary_check\data\geo_rewrites\july.txt,True,3285
9,geo_rewrite_2023_2024,Rimowa,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\supplementary_check\data\geo_rewrites\rimowa.txt,True,1647


## Compute citation, rank, feature, and PAIS-style metrics


In [7]:
# =============================
# Citation and feature extraction
# =============================

PRODUCT_PATTERN = re.compile(r"\[(Away|BR|Briggs\s*(?:&|and)\s*Riley|Briggs\s*Riley|Century|Hartmann|July|Rimowa|RIMOWA)\]", flags=re.IGNORECASE)

CANONICAL = {
    "away": "Away",
    "br": "BR",
    "briggs & riley": "BR",
    "briggs and riley": "BR",
    "briggs riley": "BR",
    "century": "Century",
    "hartmann": "Century",
    "july": "July",
    "rimowa": "Rimowa",
}

# Feature categories are intentionally broad and product-page-oriented.
# The added categories help later prompt sets 11-20.
FEATURE_PATTERNS = {
    "trip_length": r"\b(3-5|3–5|4-6|4–6|2-5|2–5|day trip|trip length|overnight|weekend|getaway|outfits?)\b",
    "overhead_bin_or_airline_fit": r"\b(overhead|carry-on|carry on|sizer|airline|domestic airlines?|flight|bin|gate-check|gate check|restrictions?)\b",
    "dimensions": r"\b(dimensions?|measurements?|inches|inch|in\.|cm|height|width|depth|linear|exterior|interior)\b",
    "weight": r"\b(weight|lbs?|pounds?|kg|lightweight|lighter|lift)\b",
    "capacity_volume": r"\b(capacity|volume|liters?|litres?|L\b|packing space|room|extra room|capacity-to-weight)\b",
    "material_shell": r"\b(polycarbonate|PC/ABS|shell|hard shell|hardside|hard-sided|micro diamond|recycled materials?)\b",
    "durability_protection": r"\b(durable|durability|protect|protection|impact|cracking|corner|guard|armor|scuff|scratch|resistant|resilience|shock|rough handling)\b",
    "wheels_mobility": r"\b(wheels?|spinner|rolling|roll|glide|gliding|maneuver|manoeuvre|mobility|PrecisionGlide|MagnaTrac|Hinomoto|360|crowded|streets?)\b",
    "handle": r"\b(handle|telescopic|trolley|PowerScope|Contour Grip|grip|cushioned|soft-grip|GEL|ergonomic)\b",
    "lock_security": r"\b(TSA|lock|security|secure|Travel Sentry|combination|SECURITECH|zip|zipper|intrusion)\b",
    "expansion": r"\b(expand|expandable|expansion|2-inch|2 inch|extra space|overpacker|overpackers)\b",
    "packing_organization": r"\b(compartment|compression|pocket|divider|strap|laundry|shoe|mesh|organize|organization|clamshell|butterfly|valuables|pouch|small items?)\b",
    "warranty_trial_return": r"\b(warranty|trial|return|returns|lifetime|10-year|100-day|promise|coverage|reassurance|purchase confidence)\b",
    "price_value": r"\b(price|priced|cost|lower-priced|budget|value|sale|discount|investment)\b",
    "electronics_convenience": r"\b(USB|USB-A|USB-C|charging|charger|battery|power bank|phone|electronics?|device|tracking|tracker|tech-heavy)\b",
    "cleaning_care": r"\b(clean|cleaning|wipe|wipe-clean|stain|water-resistant|water resistant|scratch|scuff|finish|marks?|dust bag|care)\b",
    "sustainability": r"\b(sustainable|sustainability|responsible|recycled|B Corporation|B Corp|eco-friendly|eco|materials?)\b",
    "comparison_ready_evidence": r"\b(specs?|specifications?|details?|feature details?|trust signals?|buyer caveats?|compare|comparison|clear information|information quality)\b",
    "tradeoff_or_limitation": r"\b(limitation|limit|trade-off|tradeoff|may not|does not|caution|check|vary|not provide|less detail|less complete|harder to compare|however|but|weaker choice)\b",
}

def canonical_product(label: str) -> str:
    return CANONICAL.get(label.lower(), label)

def citation_sequence(text: str):
    return [canonical_product(m.group(1)) for m in PRODUCT_PATTERN.finditer(text or "")]

# Provider-style mention metrics. These parse natural brand/product mentions after
# removing controlled bracket citations such as [Away]. Citation metrics remain
# separate from plain brand-mention metrics.
PRODUCT_MENTION_ALIASES = {
    "Away": ["Away"],
    "BR": ["Briggs & Riley", "Briggs and Riley", "Briggs Riley", "Baseline", "Essential 22"],
    "Century": ["Century", "Hartmann", "Century Deluxe"],
    "July": ["July"],
    "Rimowa": ["RIMOWA", "Rimowa", "Essential Lite"],
}

MENTION_CANONICAL = {}
for _canon, _aliases in PRODUCT_MENTION_ALIASES.items():
    for _alias in _aliases:
        MENTION_CANONICAL[_alias.lower()] = _canon

# Match longer aliases first so "Platinum Elite" is not split into smaller fragments.
MENTION_PATTERN = re.compile(
    r"\b(" + "|".join(re.escape(a) for aliases in PRODUCT_MENTION_ALIASES.values() for a in sorted(aliases, key=len, reverse=True)) + r")\b",
    flags=re.IGNORECASE,
)

POSITIVE_CONTEXT_PATTERN = re.compile(
    r"\b(best|strong|recommend(?:ed)?|useful|durable|smooth|secure|clear|confidence|better|stand(?:s)? out|excellent|ideal|good|reliable|premium|spacious|lightweight|protect(?:s|ive|ion)?|organized|convenient|easy|fits?|warranty|trial|trusted|effortless|precise|room|capacity)\b",
    flags=re.IGNORECASE,
)
NEGATIVE_CONTEXT_PATTERN = re.compile(
    r"\b(caution|limitation|limited|not|may not|does not|doesn't|heavy|larger|weaker|less|risk|gate-check|gate check|check|expensive|unclear|missing|lacks?|trade[- ]?off|downside|fully expanded|may exceed|harder)\b",
    flags=re.IGNORECASE,
)

def answer_without_product_citations(text: str) -> str:
    return PRODUCT_PATTERN.sub(" ", text or "")

def canonical_mention(label: str) -> str:
    return MENTION_CANONICAL.get((label or "").lower(), label)

def brand_mention_sequence(text: str):
    clean_text = answer_without_product_citations(text)
    return [canonical_mention(m.group(1)) for m in MENTION_PATTERN.finditer(clean_text or "")]

def product_mentioned_in_text(text: str, product: str) -> bool:
    return product in set(brand_mention_sequence(text))

def mentioned_sentences_for_product(answer: str, product: str):
    sentences = []
    for sent in split_sentences(answer_without_product_citations(answer)):
        if product_mentioned_in_text(sent, product):
            sentences.append(sent)
    return sentences

def simple_brand_sentiment(context_text: str):
    # Lightweight dictionary proxy for answer-context tone. This is descriptive only,
    # not a substitute for a validated sentiment model.
    pos = len(POSITIVE_CONTEXT_PATTERN.findall(context_text or ""))
    neg = len(NEGATIVE_CONTEXT_PATTERN.findall(context_text or ""))
    denom = pos + neg
    score = (pos - neg) / denom if denom else 0.0
    return float(score), int(pos), int(neg)


def split_sentences(text: str):
    parts = re.split(r"(?<=[.!?])\s+", (text or "").strip())
    return [p.strip() for p in parts if p.strip()]

def cited_sentences_for_product(answer: str, product: str):
    sentences = []
    for sent in split_sentences(answer):
        if product in set(citation_sequence(sent)):
            sentences.append(sent)
    return sentences

def categories_in_text(text: str):
    found = []
    for cat, pat in FEATURE_PATTERNS.items():
        if re.search(pat, text or "", flags=re.IGNORECASE):
            found.append(cat)
    return sorted(set(found))

def product_feature_categories(answer: str, product: str):
    cats = set()
    for sent in cited_sentences_for_product(answer, product):
        cats.update(categories_in_text(sent))
    return sorted(cats)

def word_count(text: str) -> int:
    return len(re.findall(r"\b\w+\b", text or ""))

def simple_ngram_overlap(a: str, b: str, n: int = 2) -> float:
    def toks(x):
        return re.findall(r"\b[a-zA-Z0-9]+\b", (x or "").lower())
    ta, tb = toks(a), toks(b)
    if len(ta) < n or len(tb) < n:
        return 0.0
    nga = set(tuple(ta[i:i+n]) for i in range(len(ta)-n+1))
    ngb = set(tuple(tb[i:i+n]) for i in range(len(tb)-n+1))
    if not nga or not ngb:
        return 0.0
    return len(nga & ngb) / len(ngb)

def tfidf_similarity_to_product_text(product_text: str, cited_answer_text: str) -> float:
    if not SKLEARN_AVAILABLE:
        return 0.0
    if not product_text.strip() or not cited_answer_text.strip():
        return 0.0
    try:
        vec = TfidfVectorizer(stop_words="english", ngram_range=(1, 2), min_df=1)
        X = vec.fit_transform([product_text, cited_answer_text])
        return float(cosine_similarity(X[0:1], X[1:2])[0, 0])
    except Exception:
        return 0.0

def build_question_product_metrics(parsed_df: pd.DataFrame, mode: str) -> pd.DataFrame:
    rows = []

    for _, row in parsed_df.iterrows():
        source_file = row["source_file"]
        source_path = row.get("source_path", "")
        run_type = row["run_type"]
        focal_geo_product = row.get("focal_geo_product", None)
        qid = int(row["question_id"])
        answer = row["answer_text"]
        seq = citation_sequence(answer)
        total_product_ref_count = len(seq)
        mention_seq = brand_mention_sequence(answer)
        total_brand_mention_count = len(mention_seq)

        unique_order = []
        for p in seq:
            if p not in unique_order:
                unique_order.append(p)

        first_unique = unique_order[0] if unique_order else None

        unique_mention_order = []
        for p in mention_seq:
            if p not in unique_mention_order:
                unique_mention_order.append(p)
        first_mentioned_unique = unique_mention_order[0] if unique_mention_order else None

        for product in PRODUCTS:
            if mode == "baseline":
                version_status = "current_2026"
            else:
                version_status = "geo_rewrite_2023_2024" if product == focal_geo_product else "current_2026"

            ref_count = seq.count(product)
            cited = ref_count > 0
            first_idx = seq.index(product) + 1 if cited else np.nan
            first_score = 1 / first_idx if cited else 0.0
            citation_share_of_voice = ref_count / total_product_ref_count if total_product_ref_count else 0.0
            citation_position_proxy = first_idx if cited else N_PRODUCTS + 1
            citation_position_score = max((N_PRODUCTS + 1 - citation_position_proxy) / N_PRODUCTS, 0)

            brand_mention_count = mention_seq.count(product)
            brand_mentioned = brand_mention_count > 0
            first_mention_idx = mention_seq.index(product) + 1 if brand_mentioned else np.nan
            first_mention_score = 1 / first_mention_idx if brand_mentioned else 0.0
            mention_share_of_voice = brand_mention_count / total_brand_mention_count if total_brand_mention_count else 0.0
            mention_position_proxy = unique_mention_order.index(product) + 1 if product in unique_mention_order else N_PRODUCTS + 1
            mention_position_score = max((N_PRODUCTS + 1 - mention_position_proxy) / N_PRODUCTS, 0)
            average_mention_position_proxy = mention_position_proxy

            recommendation_rank_proxy = unique_order.index(product) + 1 if product in unique_order else N_PRODUCTS + 1
            rank_score_proxy = max((N_PRODUCTS + 1 - recommendation_rank_proxy) / N_PRODUCTS, 0)

            cited_sents = cited_sentences_for_product(answer, product)
            mention_sents = mentioned_sentences_for_product(answer, product)
            cited_answer_text = " ".join(cited_sents)
            mentioned_answer_text = " ".join(mention_sents)
            product_text = product_texts.get((version_status, product), "")

            feature_cats = product_feature_categories(answer, product)
            feature_coverage = len(feature_cats) / len(FEATURE_PATTERNS)
            tfidf_sim = tfidf_similarity_to_product_text(product_text, cited_answer_text) if cited else 0.0
            ngram_overlap = simple_ngram_overlap(product_text, cited_answer_text, n=2) if cited else 0.0
            brand_context_text = " ".join([cited_answer_text, mentioned_answer_text]).strip()
            brand_context_sentiment_score, brand_context_positive_hits, brand_context_negative_hits = simple_brand_sentiment(brand_context_text)

            ref_component = min(ref_count / 3, 1)
            first_component = first_score

            # Product Answer Influence Score, adapted for this benchmark.
            # This is an observational proxy, not a causal or attention-based metric.
            pais = (
                0.25 * ref_component
                + 0.20 * first_component
                + 0.20 * feature_coverage
                + 0.20 * tfidf_sim
                + 0.15 * ngram_overlap
            )

            rows.append({
                "comparison": COMPARISON_NAME,
                "mode": mode,
                "source_file": source_file,
                "source_path": source_path,
                "run_type": run_type,
                "focal_geo_product": focal_geo_product,
                "version_status_for_product": version_status,
                "prompt_set_start": row.get("prompt_set_start", np.nan),
                "prompt_set_end": row.get("prompt_set_end", np.nan),
                "prompt_set_label": row.get("prompt_set_label", ""),
                "question_id": qid,
                "product": product,
                "cited": int(cited),
                "ref_count": ref_count,
                "first_citation_index": first_idx,
                "first_citation_score": first_score,
                "citation_position_proxy": citation_position_proxy,
                "citation_position_score": citation_position_score,
                "citation_share_of_voice": citation_share_of_voice,
                "recommendation_rank_proxy": recommendation_rank_proxy,
                "rank_score_proxy": rank_score_proxy,
                "top1_cited_flag": int(product == first_unique),
                "brand_mentioned": int(brand_mentioned),
                "brand_mention_count": brand_mention_count,
                "first_mention_index": first_mention_idx,
                "first_mention_score": first_mention_score,
                "mention_share_of_voice": mention_share_of_voice,
                "mention_position_proxy": mention_position_proxy,
                "mention_position_score": mention_position_score,
                "average_mention_position_proxy": average_mention_position_proxy,
                "top1_mentioned_flag": int(product == first_mentioned_unique),
                "feature_categories": ";".join(feature_cats),
                "feature_count": len(feature_cats),
                "feature_coverage": feature_coverage,
                "tfidf_product_answer_similarity": tfidf_sim,
                "bigram_overlap_product_answer": ngram_overlap,
                "pais_product_answer_influence_score": pais,
                "brand_context_sentiment_score": brand_context_sentiment_score,
                "brand_context_positive_hits": brand_context_positive_hits,
                "brand_context_negative_hits": brand_context_negative_hits,
                "answer_word_count": word_count(answer),
                "citation_sequence": " > ".join(seq),
                "cited_answer_sentences": cited_answer_text,
                "mentioned_answer_sentences": mentioned_answer_text,
            })

    return pd.DataFrame(rows)

baseline_qp_metrics = build_question_product_metrics(baseline_answers, mode="baseline")
geo_qp_metrics = build_question_product_metrics(geo_answers, mode="one_product_geo_against_current")

display(baseline_qp_metrics.head(15))
display(geo_qp_metrics.head(15))


,comparison,mode,source_file,source_path,run_type,focal_geo_product,version_status_for_product,prompt_set_start,prompt_set_end,prompt_set_label,...,tfidf_product_answer_similarity,bigram_overlap_product_answer,pais_product_answer_influence_score,brand_context_sentiment_score,brand_context_positive_hits,brand_context_negative_hits,answer_word_count,citation_sequence,cited_answer_sentences,mentioned_answer_sentences
0,03_2026_current_vs_2023_2024_geo_rewrite,baseline,baseline_curr/baseline_1.txt,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\supplementary_check\results\shopping_answers\baseline_curr\baseline_1.txt,baseline_all_current_2026,None,current_2026,1,5,1-5,...,0.153288,0.320988,0.550736,1.000000,8,0,120,Away > Away > July,"[Away]\nIt also has a 41 L capacity, a durable lightweight 100% polycarbonate shell, a TSA-approved combination lock, smooth-gliding wheels, and an interior compression system ...","For a 3-5 day trip, I would recommend the Away Carry-On because its product page directly says it is generally best for 3-5 day trips and sized for about 5-7 full outfits."
1,03_2026_current_vs_2023_2024_geo_rewrite,baseline,baseline_curr/baseline_1.txt,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\supplementary_check\results\shopping_answers\baseline_curr\baseline_1.txt,baseline_all_current_2026,None,current_2026,1,5,1-5,...,0.000000,0.000000,0.000000,0.000000,0,0,120,Away > Away > July,,
2,03_2026_current_vs_2023_2024_geo_rewrite,baseline,baseline_curr/baseline_1.txt,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\supplementary_check\results\shopping_answers\baseline_curr\baseline_1.txt,baseline_all_current_2026,None,current_2026,1,5,1-5,...,0.000000,0.000000,0.000000,0.000000,0,0,120,Away > Away > July,,
3,03_2026_current_vs_2023_2024_geo_rewrite,baseline,baseline_curr/baseline_1.txt,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\supplementary_check\results\shopping_answers\baseline_curr\baseline_1.txt,baseline_all_current_2026,None,current_2026,1,5,1-5,...,0.085846,0.000000,0.167169,1.000000,2,0,120,Away > Away > July,[July],"July is a good alternative if you want a little more packing capacity because it is listed for 5-7 days, 6-8 outfits, 42 L, and includes a Y-Strap compression system, but it ma..."
4,03_2026_current_vs_2023_2024_geo_rewrite,baseline,baseline_curr/baseline_1.txt,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\supplementary_check\results\shopping_answers\baseline_curr\baseline_1.txt,baseline_all_current_2026,None,current_2026,1,5,1-5,...,0.000000,0.000000,0.000000,0.000000,0,0,120,Away > Away > July,,
5,03_2026_current_vs_2023_2024_geo_rewrite,baseline,baseline_curr/baseline_1.txt,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\supplementary_check\results\shopping_answers\baseline_curr\baseline_1.txt,baseline_all_current_2026,None,current_2026,1,5,1-5,...,0.102574,0.000000,0.153848,1.000000,1,0,157,BR > BR > July > Away,[Away],"Away is another strong hardside choice because its handles, wheels, zippers, and shell are described as rigorously tested, including handle pulls, simulated city-block rolling,..."
6,03_2026_current_vs_2023_2024_geo_rewrite,baseline,baseline_curr/baseline_1.txt,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\supplementary_check\results\shopping_answers\baseline_curr\baseline_1.txt,baseline_all_current_2026,None,current_2026,1,5,1-5,...,0.074024,0.291667,0.467327,1.000000,7,0,157,BR > BR > July > Away,"[Briggs & Riley]\nIts lifetime guarantee is especially confidence-building because the page says that if the bag is ever damaged, even by an airline, Briggs & Riley will repair...","The Briggs & Riley Baseline Essential 22"" Carry-On Expandable Spinner seems strongest if durability and long-term protection matter most because it is made from premium ultra-d..."
7,03_2026_current_vs_2023_2024_geo_rewrite,baseline,baseline_curr/baseline_1.txt,C:\Users\junse\Documents

,comparison,mode,source_file,source_path,run_type,focal_geo_product,version_status_for_product,prompt_set_start,prompt_set_end,prompt_set_label,...,tfidf_product_answer_similarity,bigram_overlap_product_answer,pais_product_answer_influence_score,brand_context_sentiment_score,brand_context_positive_hits,brand_context_negative_hits,answer_word_count,citation_sequence,cited_answer_sentences,mentioned_answer_sentences
0,03_2026_current_vs_2023_2024_geo_rewrite,one_product_geo_against_current,03_2026_current_vs_2023_2024_geo_rewrite/away_1.txt,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\supplementary_check\results\shopping_answers\03_2026_current_vs_2023_2024_geo_rewrite\away_1.txt,one_product_geo_rewrite_against_current,Away,geo_rewrite_2023_2024,1,5,1-5,...,0.154849,0.193878,0.531981,0.800000,9,1,151,Away > Away > July,"[Away] It also has a 39.8 L capacity, a 7.5 lb weight, a durable lightweight 100% polycarbonate shell, 360° smooth-gliding wheels, an interior compression system, mesh pockets,...","I would recommend Away for a 3-5 day trip because its page specifically says The Carry-On is made for quick trips of 3-5 days, fits 5-7 outfits, and is sized to fit in all over..."
1,03_2026_current_vs_2023_2024_geo_rewrite,one_product_geo_against_current,03_2026_current_vs_2023_2024_geo_rewrite/away_1.txt,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\supplementary_check\results\shopping_answers\03_2026_current_vs_2023_2024_geo_rewrite\away_1.txt,one_product_geo_rewrite_against_current,Away,current_2026,1,5,1-5,...,0.000000,0.000000,0.000000,0.000000,0,0,151,Away > Away > July,,
2,03_2026_current_vs_2023_2024_geo_rewrite,one_product_geo_against_current,03_2026_current_vs_2023_2024_geo_rewrite/away_1.txt,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\supplementary_check\results\shopping_answers\03_2026_current_vs_2023_2024_geo_rewrite\away_1.txt,one_product_geo_rewrite_against_current,Away,current_2026,1,5,1-5,...,0.000000,0.000000,0.000000,0.000000,0,0,151,Away > Away > July,,
3,03_2026_current_vs_2023_2024_geo_rewrite,one_product_geo_against_current,03_2026_current_vs_2023_2024_geo_rewrite/away_1.txt,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\supplementary_check\results\shopping_answers\03_2026_current_vs_2023_2024_geo_rewrite\away_1.txt,one_product_geo_rewrite_against_current,Away,current_2026,1,5,1-5,...,0.085846,0.000000,0.167169,0.333333,2,1,151,Away > Away > July,[July],"July is also a strong option if you want a slightly larger carry-on because it is listed for 5-7 days, 6-8 outfits, 42 L capacity, and carry-on approval for all International s..."
4,03_2026_current_vs_2023_2024_geo_rewrite,one_product_geo_against_current,03_2026_current_vs_2023_2024_geo_rewrite/away_1.txt,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\supplementary_check\results\shopping_answers\03_2026_current_vs_2023_2024_geo_rewrite\away_1.txt,one_product_geo_rewrite_against_current,Away,current_2026,1,5,1-5,...,0.000000,0.000000,0.000000,0.000000,0,0,151,Away > Away > July,,
5,03_2026_current_vs_2023_2024_geo_rewrite,one_product_geo_against_current,03_2026_current_vs_2023_2024_geo_rewrite/away_1.txt,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\supplementary_check\results\shopping_answers\03_2026_current_vs_2023_2024_geo_rewrite\away_1.txt,one_product_geo_rewrite_against_current,Away,geo_rewrite_2023_2024,1,5,1-5,...,0.013476,0.000000,0.147081,-0.333333,1,2,190,July > BR > July > BR > Away,"[Away] The provided pages give brand claims and feature lists, but they do not provide independent drop-test results or side-by-side durability testing, so the durability recom...","Away is also credible for durability because it uses a rigorously tested 100% polycarbonate hard shell, ergonomic reverse-coil zippers, and a limited warranty covering function..."
6,03_2026_current_vs_2023_2024_geo_rewrite,one_product_geo_against_current,03_2026_current_vs_2023_2024_

## Match GEO replacement focal rows to all-current baseline by product and question

This is the key step: the delta is computed only after matching the focal GEO-rewrite replacement product row to the same product/question in the all-current 2026 baseline.


In [8]:
# =============================
# Matched GEO-rewrite-vs-current-baseline question-product deltas
# =============================

METRIC_COLS = [
    "cited",
    "ref_count",
    "first_citation_score",
    "recommendation_rank_proxy",
    "rank_score_proxy",
    "top1_cited_flag",
    "feature_count",
    "feature_coverage",
    "tfidf_product_answer_similarity",
    "bigram_overlap_product_answer",
    "pais_product_answer_influence_score",
    "brand_mentioned",
    "brand_mention_count",
    "mention_share_of_voice",
    "citation_share_of_voice",
    "first_mention_score",
    "mention_position_proxy",
    "mention_position_score",
    "average_mention_position_proxy",
    "citation_position_proxy",
    "citation_position_score",
    "brand_context_sentiment_score",
    "brand_context_positive_hits",
    "brand_context_negative_hits",
]

geo_focal_rows = geo_qp_metrics[
    geo_qp_metrics["product"] == geo_qp_metrics["focal_geo_product"]
].copy()

current_lookup_cols = [
    "product",
    "question_id",
    "source_file",
    "prompt_set_label",
] + METRIC_COLS

current_lookup = baseline_qp_metrics[current_lookup_cols].copy()

# If multiple all-current baseline files contain the same product/question_id, average them.
# This supports repeated baseline-current runs while keeping the comparison stable.
agg_dict = {col: "mean" for col in METRIC_COLS}
agg_dict.update({
    "source_file": lambda x: ";".join(sorted(set(map(str, x)))),
    "prompt_set_label": lambda x: join_prompt_set_labels(x),
})
current_lookup = (
    current_lookup
    .groupby(["product", "question_id"], as_index=False)
    .agg(agg_dict)
)

matched = geo_focal_rows.merge(
    current_lookup,
    on=["product", "question_id"],
    how="left",
    suffixes=("_geo", "_current"),
)

matched["current_baseline_match_found"] = matched["source_file_current"].notna()

missing_current_matches = matched[~matched["current_baseline_match_found"]].copy()
geo_question_ids = set(map(int, geo_focal_rows["question_id"].unique()))
current_question_ids = set(map(int, current_lookup["question_id"].unique()))
matched_question_ids = set(map(int, matched[matched["current_baseline_match_found"]]["question_id"].unique()))
missing_current_question_ids = sorted(geo_question_ids - current_question_ids)
missing_geo_question_ids_vs_expected = sorted(set(EXPECTED_QUESTION_IDS) - geo_question_ids)
missing_current_question_ids_vs_expected = sorted(set(EXPECTED_QUESTION_IDS) - current_question_ids)

matching_coverage_summary = pd.DataFrame([{
    "n_geo_focal_rows": len(geo_focal_rows),
    "n_matched_rows": int(matched["current_baseline_match_found"].sum()),
    "n_unmatched_geo_rows": int((~matched["current_baseline_match_found"]).sum()),
    "geo_replacement_question_ids": ",".join(map(str, sorted(geo_question_ids))),
    "current_baseline_question_ids": ",".join(map(str, sorted(current_question_ids))),
    "matched_question_ids": ",".join(map(str, sorted(matched_question_ids))),
    "missing_current_baseline_question_ids_for_geo": ",".join(map(str, missing_current_question_ids)),
    "missing_geo_question_ids_vs_expected": ",".join(map(str, missing_geo_question_ids_vs_expected)),
    "missing_current_question_ids_vs_expected": ",".join(map(str, missing_current_question_ids_vs_expected)),
}])

print("Matching coverage summary:")
display(matching_coverage_summary)

if not missing_current_matches.empty:
    print("WARNING: Some GEO focal rows do not have matching all-current baseline question IDs.")
    display(
        missing_current_matches[
            ["source_file_geo", "focal_geo_product", "product", "question_id", "prompt_set_label_geo"]
        ].sort_values(["product", "question_id", "source_file_geo"])
    )

if missing_geo_question_ids_vs_expected:
    print("NOTE: These expected question IDs are not present in included GEO replacement files:", missing_geo_question_ids_vs_expected)
    print("This is okay if you currently only have _1 and _6 files. Once _11 and _16 files are added for each product, this list should become empty.")

if (not ALLOW_PARTIAL_MATCH) and missing_current_question_ids:
    raise ValueError(
        "Cannot compute a full multi-prompt GEO-vs-current delta because some GEO question IDs do not have matching all-current baseline answers. "
        f"Missing all-current baseline question IDs for GEO files: {missing_current_question_ids}. "
        "Add all-current baseline answer files for those questions to the baseline_curr folder, or set ALLOW_PARTIAL_MATCH = True if you intentionally want a partial matched analysis."
    )

matched_complete = matched[matched["current_baseline_match_found"]].copy()

if matched_complete.empty:
    raise ValueError("No matched GEO-vs-current rows were found. Check question IDs and baseline-current files.")

for col in METRIC_COLS:
    matched_complete[f"delta_{col}"] = matched_complete[f"{col}_geo"] - matched_complete[f"{col}_current"]

# More readable aliases for downstream tables.
matched_complete["delta_citation_rate"] = matched_complete["delta_cited"]
matched_complete["delta_mean_ref_count"] = matched_complete["delta_ref_count"]
matched_complete["rank_improvement_geo_minus_current"] = (
    matched_complete["recommendation_rank_proxy_current"] - matched_complete["recommendation_rank_proxy_geo"]
)
matched_complete["delta_top1_cited_share"] = matched_complete["delta_top1_cited_flag"]
matched_complete["delta_pais"] = matched_complete["delta_pais_product_answer_influence_score"]

print("GEO focal rows:", len(geo_focal_rows))
print("Matched rows used for delta:", len(matched_complete))
display(matched_complete.head(30))


Matching coverage summary:


,n_geo_focal_rows,n_matched_rows,n_unmatched_geo_rows,geo_replacement_question_ids,current_baseline_question_ids,matched_question_ids,missing_current_baseline_question_ids_for_geo,missing_geo_question_ids_vs_expected,missing_current_question_ids_vs_expected
0,95,95,0,"1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20","1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20","1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20",,,


GEO focal rows: 95
Matched rows used for delta: 95


,comparison,mode,source_file_geo,source_path,run_type,focal_geo_product,version_status_for_product,prompt_set_start,prompt_set_end,prompt_set_label_geo,...,delta_citation_position_proxy,delta_citation_position_score,delta_brand_context_sentiment_score,delta_brand_context_positive_hits,delta_brand_context_negative_hits,delta_citation_rate,delta_mean_ref_count,rank_improvement_geo_minus_current,delta_top1_cited_share,delta_pais
0,03_2026_current_vs_2023_2024_geo_rewrite,one_product_geo_against_current,03_2026_current_vs_2023_2024_geo_rewrite/away_1.txt,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\supplementary_check\results\shopping_answers\03_2026_current_vs_2023_2024_geo_rewrite\away_1.txt,one_product_geo_rewrite_against_current,Away,geo_rewrite_2023_2024,1,5,1-5,...,0.0,0.0,-0.200000,1.0,1.0,0.0,0.0,0.0,0.0,-0.018754
1,03_2026_current_vs_2023_2024_geo_rewrite,one_product_geo_against_current,03_2026_current_vs_2023_2024_geo_rewrite/away_1.txt,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\supplementary_check\results\shopping_answers\03_2026_current_vs_2023_2024_geo_rewrite\away_1.txt,one_product_geo_rewrite_against_current,Away,geo_rewrite_2023_2024,1,5,1-5,...,1.0,-0.2,-1.333333,0.0,2.0,0.0,0.0,0.0,0.0,-0.006767
2,03_2026_current_vs_2023_2024_geo_rewrite,one_product_geo_against_current,03_2026_current_vs_2023_2024_geo_rewrite/away_1.txt,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\supplementary_check\results\shopping_answers\03_2026_current_vs_2023_2024_geo_rewrite\away_1.txt,one_product_geo_rewrite_against_current,Away,geo_rewrite_2023_2024,1,5,1-5,...,-5.0,1.0,0.000000,10.0,0.0,1.0,2.0,5.0,1.0,0.497554
3,03_2026_current_vs_2023_2024_geo_rewrite,one_product_geo_against_current,03_2026_current_vs_2023_2024_geo_rewrite/away_1.txt,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\supplementary_check\results\shopping_answers\03_2026_current_vs_2023_2024_geo_rewrite\away_1.txt,one_product_geo_rewrite_against_current,Away,geo_rewrite_2023_2024,1,5,1-5,...,1.0,-0.2,0.000000,1.0,0.0,0.0,-1.0,0.0,0.0,-0.093466
4,03_2026_current_vs_2023_2024_geo_rewrite,one_product_geo_against_current,03_2026_current_vs_2023_2024_geo_rewrite/away_1.txt,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\supplementary_check\results\shopping_answers\03_2026_current_vs_2023_2024_geo_rewrite\away_1.txt,one_product_geo_rewrite_against_current,Away,geo_rewrite_2023_2024,1,5,1-5,...,0.0,0.0,-0.351515,1.0,3.0,0.0,1.0,0.0,0.0,0.175155
5,03_2026_current_vs_2023_2024_geo_rewrite,one_product_geo_against_current,03_2026_current_vs_2023_2024_geo_rewrite/away_11.txt,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\supplementary_check\results\shopping_answers\03_2026_current_vs_2023_2024_geo_rewrite\away_11.txt,one_product_geo_rewrite_against_current,Away,geo_rewrite_2023_2024,11,15,11-15,...,0.0,0.0,0.666667,0.0,-2.0,0.0,-1.0,0.0,0.0,-0.088023
6,03_2026_current_vs_2023_2024_geo_rewrite,one_product_geo_against_current,03_2026_current_vs_2023_2024_geo_rewrite/away_11.txt,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\supplementary_check\results\shopping_answers\03_2026_current_vs_2023_2024_geo_rewrite\away_11.txt,one_product_geo_rewrite_against_current,Away,geo_rewrite_2023_2024,11,15,11-15,...,0.0,0.0,0.000000,0.0,0.0,0.0,-1.0,0.0,0.0,-0.056201
7,03_2026_current_vs_2023_2024_geo_rewrite,one_product_geo_against_current,03_2026_current_vs_2023_2024_geo_rewrite/away_11.txt,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\supplementary_check\results\shopping_answers\03_2026_current_vs_2023_2024_geo_rewrite\away_11.txt,one_product_geo_rewrite_against_current,Away,geo_rewrite_2023_2024,11,15,11-15,...,0.0,0.0,-0.928571,-3.0,4.0,0.0,-2.0,0.0,0.0,-0.080545
8,03_2026_current_vs_2023_2024_geo_rewrite,one_product_geo_against_current,03_2026_current_vs_2023_2024_geo_rewrite/away_11.txt,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\supp

In [9]:
# =============================
# Product-level summaries from matched rows
# =============================

def summarize_side_from_matched(matched_df: pd.DataFrame, side: str, label: str) -> pd.DataFrame:
    suffix = "_geo" if side == "geo" else "_current"

    temp = pd.DataFrame({
        "product": matched_df["product"],
        "question_id": matched_df["question_id"],
        "source_file_geo": matched_df["source_file_geo"],
        "source_file_current": matched_df["source_file_current"],
        "prompt_set_label_geo": matched_df["prompt_set_label_geo"],
        "cited": matched_df[f"cited{suffix}"],
        "ref_count": matched_df[f"ref_count{suffix}"],
        "first_citation_score": matched_df[f"first_citation_score{suffix}"],
        "recommendation_rank_proxy": matched_df[f"recommendation_rank_proxy{suffix}"],
        "rank_score_proxy": matched_df[f"rank_score_proxy{suffix}"],
        "top1_cited_flag": matched_df[f"top1_cited_flag{suffix}"],
        "feature_count": matched_df[f"feature_count{suffix}"],
        "feature_coverage": matched_df[f"feature_coverage{suffix}"],
        "tfidf_product_answer_similarity": matched_df[f"tfidf_product_answer_similarity{suffix}"],
        "bigram_overlap_product_answer": matched_df[f"bigram_overlap_product_answer{suffix}"],
        "pais_product_answer_influence_score": matched_df[f"pais_product_answer_influence_score{suffix}"],
    })

    summary = (
        temp.groupby("product")
        .agg(
            n_question_contexts=("question_id", "count"),
            question_ids=("question_id", lambda x: ",".join(map(str, sorted(set(x))))),
            n_geo_source_files=("source_file_geo", "nunique"),
            n_current_source_files=("source_file_current", "nunique"),
            prompt_sets=("prompt_set_label_geo", lambda x: join_prompt_set_labels(x)),
            citation_rate=("cited", "mean"),
            mean_ref_count=("ref_count", "mean"),
            total_ref_count=("ref_count", "sum"),
            mean_first_citation_score=("first_citation_score", "mean"),
            mean_recommendation_rank_proxy=("recommendation_rank_proxy", "mean"),
            mean_rank_score_proxy=("rank_score_proxy", "mean"),
            top1_cited_share=("top1_cited_flag", "mean"),
            mean_feature_count=("feature_count", "mean"),
            mean_feature_coverage=("feature_coverage", "mean"),
            mean_tfidf_product_answer_similarity=("tfidf_product_answer_similarity", "mean"),
            mean_bigram_overlap_product_answer=("bigram_overlap_product_answer", "mean"),
            mean_pais_product_answer_influence_score=("pais_product_answer_influence_score", "mean"),
        )
        .reset_index()
    )
    summary.insert(0, "metric_source", label)
    return summary

current_product_summary_matched = summarize_side_from_matched(
    matched_complete,
    side="current",
    label="baseline_all_current_2026_matched_questions",
)

geo_product_summary_matched = summarize_side_from_matched(
    matched_complete,
    side="geo",
    label="one_product_geo_rewrite_matched_questions",
)

display(current_product_summary_matched)
display(geo_product_summary_matched)


,metric_source,product,n_question_contexts,question_ids,n_geo_source_files,n_current_source_files,prompt_sets,citation_rate,mean_ref_count,total_ref_count,mean_first_citation_score,mean_recommendation_rank_proxy,mean_rank_score_proxy,top1_cited_share,mean_feature_count,mean_feature_coverage,mean_tfidf_product_answer_similarity,mean_bigram_overlap_product_answer,mean_pais_product_answer_influence_score
0,baseline_all_current_2026_matched_questions,Away,20,"1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20",4,4,1-5;6-10;11-15;16-20,0.85,1.5,30.0,0.524167,2.70,0.66,0.350000,4.150000,0.218421,0.068373,0.075295,0.294320
1,baseline_all_current_2026_matched_questions,BR,20,"1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20",4,4,1-5;6-10;11-15;16-20,0.90,1.8,36.0,0.492500,2.65,0.67,0.300000,4.200000,0.221053,0.053308,0.153315,0.322203
2,baseline_all_current_2026_matched_questions,Century,20,"1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20",4,4,1-5;6-10;11-15;16-20,0.80,1.4,28.0,0.243452,4.00,0.40,0.050000,3.600000,0.189474,0.040336,0.028388,0.215577
3,baseline_all_current_2026_matched_questions,July,20,"1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20",4,4,1-5;6-10;11-15;16-20,0.90,1.5,30.0,0.482500,2.45,0.71,0.250000,4.100000,0.215789,0.059808,0.050654,0.280051
4,baseline_all_current_2026_matched_questions,Rimowa,15,"6,7,8,9,10,11,12,13,14,15,16,17,18,19,20",3,3,6-10;11-15;16-20,0.80,1.2,18.0,0.190265,4.80,0.24,0.066667,2.133333,0.112281,0.055377,0.004311,0.172231


,metric_source,product,n_question_contexts,question_ids,n_geo_source_files,n_current_source_files,prompt_sets,citation_rate,mean_ref_count,total_ref_count,mean_first_citation_score,mean_recommendation_rank_proxy,mean_rank_score_proxy,top1_cited_share,mean_feature_count,mean_feature_coverage,mean_tfidf_product_answer_similarity,mean_bigram_overlap_product_answer,mean_pais_product_answer_influence_score
0,one_product_geo_rewrite_matched_questions,Away,20,"1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20",4,4,1-5;6-10;11-15;16-20,0.90,1.250000,25,0.664167,2.050000,0.790000,0.500000,5.65,0.297368,0.071862,0.092150,0.324668
1,one_product_geo_rewrite_matched_questions,BR,20,"1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20",4,4,1-5;6-10;11-15;16-20,0.85,1.350000,27,0.443333,2.850000,0.630000,0.250000,4.65,0.244737,0.048186,0.081567,0.263653
2,one_product_geo_rewrite_matched_questions,Century,20,"1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20",4,4,1-5;6-10;11-15;16-20,0.55,0.900000,18,0.207341,4.450000,0.310000,0.100000,2.25,0.118421,0.036293,0.027921,0.151599
3,one_product_geo_rewrite_matched_questions,July,20,"1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20",4,4,1-5;6-10;11-15;16-20,0.90,1.250000,25,0.490833,2.400000,0.720000,0.300000,3.80,0.200000,0.062539,0.065508,0.264667
4,one_product_geo_rewrite_matched_questions,Rimowa,15,"6,7,8,9,10,11,12,13,14,15,16,17,18,19,20",3,3,6-10;11-15;16-20,0.80,0.933333,14,0.273968,4.066667,0.386667,0.133333,3.00,0.157895,0.039526,0.007941,0.173247


In [10]:
# =============================
# GEO rewrite - all-current baseline product-level delta
# =============================

def make_geo_vs_current_delta_from_matched(matched_df: pd.DataFrame) -> pd.DataFrame:
    rows = []

    for product, g in matched_df.groupby("product"):
        rows.append({
            "comparison": COMPARISON_NAME,
            "product": product,
            "matched_n_question_contexts": int(len(g)),
            "question_ids": ",".join(map(str, sorted(g["question_id"].unique()))),
            "geo_source_files": ";".join(sorted(g["source_file_geo"].unique())),
            "current_baseline_source_files": ";".join(sorted(g["source_file_current"].unique())),
            "prompt_sets": join_prompt_set_labels(g["prompt_set_label_geo"]),

            "current_citation_rate": g["cited_current"].mean(),
            "geo_citation_rate": g["cited_geo"].mean(),
            "delta_citation_rate": g["delta_cited"].mean(),

            "current_mean_ref_count": g["ref_count_current"].mean(),
            "geo_mean_ref_count": g["ref_count_geo"].mean(),
            "delta_mean_ref_count": g["delta_ref_count"].mean(),

            "current_first_citation_score": g["first_citation_score_current"].mean(),
            "geo_first_citation_score": g["first_citation_score_geo"].mean(),
            "delta_first_citation_score": g["delta_first_citation_score"].mean(),

            "current_mean_recommendation_rank_proxy": g["recommendation_rank_proxy_current"].mean(),
            "geo_mean_recommendation_rank_proxy": g["recommendation_rank_proxy_geo"].mean(),
            "rank_improvement_geo_minus_current": (
                g["recommendation_rank_proxy_current"].mean() - g["recommendation_rank_proxy_geo"].mean()
            ),

            "current_rank_score_proxy": g["rank_score_proxy_current"].mean(),
            "geo_rank_score_proxy": g["rank_score_proxy_geo"].mean(),
            "delta_rank_score_proxy": g["delta_rank_score_proxy"].mean(),

            "current_top1_cited_share": g["top1_cited_flag_current"].mean(),
            "geo_top1_cited_share": g["top1_cited_flag_geo"].mean(),
            "delta_top1_cited_share": g["delta_top1_cited_flag"].mean(),

            "current_feature_coverage": g["feature_coverage_current"].mean(),
            "geo_feature_coverage": g["feature_coverage_geo"].mean(),
            "delta_feature_coverage": g["delta_feature_coverage"].mean(),

            "current_pais": g["pais_product_answer_influence_score_current"].mean(),
            "geo_pais": g["pais_product_answer_influence_score_geo"].mean(),
            "delta_pais": g["delta_pais_product_answer_influence_score"].mean(),
        })

    delta = pd.DataFrame(rows)

    # Composite descriptive score.
    # Positive means the GEO rewrite replacement produced stronger answer behavior
    # than the all-current 2026 baseline for that product.
    delta["geo_vs_current_advantage_score"] = (
        0.25 * delta["delta_rank_score_proxy"]
        + 0.20 * delta["delta_citation_rate"]
        + 0.20 * delta["delta_first_citation_score"]
        + 0.15 * delta["delta_top1_cited_share"]
        + 0.10 * delta["delta_feature_coverage"]
        + 0.10 * delta["delta_pais"]
    )

    delta["direction_label"] = np.select(
        [
            delta["geo_vs_current_advantage_score"] > 0.05,
            delta["geo_vs_current_advantage_score"] < -0.05,
        ],
        [
            "geo_rewrite_stronger_than_current",
            "current_2026_stronger_than_geo_rewrite",
        ],
        default="similar_or_small_change",
    )

    return delta.sort_values("geo_vs_current_advantage_score", ascending=False)

geo_vs_current_delta_by_product = make_geo_vs_current_delta_from_matched(matched_complete)

display(geo_vs_current_delta_by_product)


,comparison,product,matched_n_question_contexts,question_ids,geo_source_files,current_baseline_source_files,prompt_sets,current_citation_rate,geo_citation_rate,delta_citation_rate,...,geo_top1_cited_share,delta_top1_cited_share,current_feature_coverage,geo_feature_coverage,delta_feature_coverage,current_pais,geo_pais,delta_pais,geo_vs_current_advantage_score,direction_label
0,03_2026_current_vs_2023_2024_geo_rewrite,Away,20,"1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20",03_2026_current_vs_2023_2024_geo_rewrite/away_1.txt;03_2026_current_vs_2023_2024_geo_rewrite/away_11.txt;03_2026_current_vs_2023_2024_geo_rewrite/away_16.txt;03_2026_current_vs...,baseline_curr/baseline_1.txt;baseline_curr/baseline_11.txt;baseline_curr/baseline_16.txt;baseline_curr/baseline_6.txt,1-5;6-10;11-15;16-20,0.85,0.90,0.05,...,0.500000,0.150000,0.218421,0.297368,0.078947,0.294320,0.324668,0.030349,0.103930,geo_rewrite_stronger_than_current
4,03_2026_current_vs_2023_2024_geo_rewrite,Rimowa,15,"6,7,8,9,10,11,12,13,14,15,16,17,18,19,20",03_2026_current_vs_2023_2024_geo_rewrite/rimowa_11.txt;03_2026_current_vs_2023_2024_geo_rewrite/rimowa_16.txt;03_2026_current_vs_2023_2024_geo_rewrite/rimowa_6.txt,baseline_curr/baseline_11.txt;baseline_curr/baseline_16.txt;baseline_curr/baseline_6.txt,6-10;11-15;16-20,0.80,0.80,0.00,...,0.133333,0.066667,0.112281,0.157895,0.045614,0.172231,0.173247,0.001016,0.068070,geo_rewrite_stronger_than_current
3,03_2026_current_vs_2023_2024_geo_rewrite,July,20,"1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20",03_2026_current_vs_2023_2024_geo_rewrite/july_1.txt;03_2026_current_vs_2023_2024_geo_rewrite/july_11.txt;03_2026_current_vs_2023_2024_geo_rewrite/july_16.txt;03_2026_current_vs...,baseline_curr/baseline_1.txt;baseline_curr/baseline_11.txt;baseline_curr/baseline_16.txt;baseline_curr/baseline_6.txt,1-5;6-10;11-15;16-20,0.90,0.90,0.00,...,0.300000,0.050000,0.215789,0.200000,-0.015789,0.280051,0.264667,-0.015384,0.008549,similar_or_small_change
1,03_2026_current_vs_2023_2024_geo_rewrite,BR,20,"1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20",03_2026_current_vs_2023_2024_geo_rewrite/br_1.txt;03_2026_current_vs_2023_2024_geo_rewrite/br_11.txt;03_2026_current_vs_2023_2024_geo_rewrite/br_16.txt;03_2026_current_vs_2023_...,baseline_curr/baseline_1.txt;baseline_curr/baseline_11.txt;baseline_curr/baseline_16.txt;baseline_curr/baseline_6.txt,1-5;6-10;11-15;16-20,0.90,0.85,-0.05,...,0.250000,-0.050000,0.221053,0.244737,0.023684,0.322203,0.263653,-0.058550,-0.040820,similar_or_small_change
2,03_2026_current_vs_2023_2024_geo_rewrite,Century,20,"1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20",03_2026_current_vs_2023_2024_geo_rewrite/century_1.txt;03_2026_current_vs_2023_2024_geo_rewrite/century_11.txt;03_2026_current_vs_2023_2024_geo_rewrite/century_16.txt;03_2026_c...,baseline_curr/baseline_1.txt;baseline_curr/baseline_11.txt;baseline_curr/baseline_16.txt;baseline_curr/baseline_6.txt,1-5;6-10;11-15;16-20,0.80,0.55,-0.25,...,0.100000,0.050000,0.189474,0.118421,-0.071053,0.215577,0.151599,-0.063978,-0.085725,current_2026_stronger_than_geo_rewrite


In [11]:
# =============================
# Overall summary
# =============================

overall_geo_vs_current_summary = pd.DataFrame([{
    "comparison": COMPARISON_NAME,
    "purpose": "Directly compare all-current 2026 baseline answers against one-product 2023/2024 GEO-style rewrite replacements",
    "interpretation": "Direct current-vs-hypothetical-GEO comparison",
    "baseline_current_dir_fallback": str(BASELINE_CURRENT_DIR),
    "baseline_current_discovered_paths": " | ".join(map(str, BASELINE_CURRENT_DISCOVERED_PATHS)),
    "comparison_dir": str(COMPARISON_DIR),
    "n_baseline_current_files": len(BASELINE_CURRENT_FILES),
    "n_included_geo_replacement_files": len(TXT_FILES),
    "n_ignored_txt_files": len(IGNORED_TXT_FILES),
    "included_question_ids": ",".join(map(str, sorted(matched_complete["question_id"].unique()))),
    "current_baseline_question_ids": ",".join(map(str, sorted(current_question_ids))),
    "geo_replacement_question_ids": ",".join(map(str, sorted(geo_question_ids))),
    "missing_current_baseline_question_ids_for_geo": ",".join(map(str, missing_current_question_ids)),
    "missing_geo_question_ids_vs_expected": ",".join(map(str, missing_geo_question_ids_vs_expected)),
    "n_products": len(geo_vs_current_delta_by_product),
    "mean_delta_citation_rate": geo_vs_current_delta_by_product["delta_citation_rate"].mean(),
    "mean_delta_rank_score_proxy": geo_vs_current_delta_by_product["delta_rank_score_proxy"].mean(),
    "mean_rank_improvement_geo_minus_current": geo_vs_current_delta_by_product["rank_improvement_geo_minus_current"].mean(),
    "mean_delta_first_citation_score": geo_vs_current_delta_by_product["delta_first_citation_score"].mean(),
    "mean_delta_top1_cited_share": geo_vs_current_delta_by_product["delta_top1_cited_share"].mean(),
    "mean_delta_feature_coverage": geo_vs_current_delta_by_product["delta_feature_coverage"].mean(),
    "mean_delta_pais": geo_vs_current_delta_by_product["delta_pais"].mean(),
    "mean_geo_vs_current_advantage_score": geo_vs_current_delta_by_product["geo_vs_current_advantage_score"].mean(),
    "n_geo_rewrite_stronger_than_current": int((geo_vs_current_delta_by_product["direction_label"] == "geo_rewrite_stronger_than_current").sum()),
    "n_current_2026_stronger_than_geo_rewrite": int((geo_vs_current_delta_by_product["direction_label"] == "current_2026_stronger_than_geo_rewrite").sum()),
    "n_similar_or_small_change": int((geo_vs_current_delta_by_product["direction_label"] == "similar_or_small_change").sum()),
}])

display(overall_geo_vs_current_summary)


,comparison,purpose,interpretation,baseline_current_dir_fallback,baseline_current_discovered_paths,comparison_dir,n_baseline_current_files,n_included_geo_replacement_files,n_ignored_txt_files,included_question_ids,...,mean_delta_rank_score_proxy,mean_rank_improvement_geo_minus_current,mean_delta_first_citation_score,mean_delta_top1_cited_share,mean_delta_feature_coverage,mean_delta_pais,mean_geo_vs_current_advantage_score,n_geo_rewrite_stronger_than_current,n_current_2026_stronger_than_geo_rewrite,n_similar_or_small_change
0,03_2026_current_vs_2023_2024_geo_rewrite,Directly compare all-current 2026 baseline answers against one-product 2023/2024 GEO-style rewrite replacements,Direct current-vs-hypothetical-GEO comparison,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\supplementary_check\results\shopping_answers\baseline_curr,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\supplementary_check\results\shopping_answers\baseline_curr,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\supplementary_check\results\shopping_answers\03_2026_current_vs_2023_2024_geo_rewrite,4,19,1,"1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20",...,0.031333,0.156667,0.029352,0.053333,0.012281,-0.021309,0.010801,2,1,2


## Provider-style answer visibility metrics

In [12]:

# =============================
# Provider-style answer visibility metrics
# =============================
# These metrics translate the product-label answer files into commercial AI-visibility language.
# source/domain share is intentionally not computed here because the prompt uses controlled
# product-label citations such as [Away], not external URL/domain citations.

PROVIDER_STYLE_METRIC_NOTE = (
    "Provider-style answer metrics added: brand_mention_rate, mention_share_of_voice, "
    "citation_share_of_voice, average_mention_position, average_citation_position, "
    "first_citation_score, sentiment context proxy, and comparison-level visibility delta. "
    "source/domain share is not available in this controlled product-label citation setting."
)

def _safe_mean(series):
    return pd.to_numeric(series, errors="coerce").mean()

def _provider_delta_rows(matched_df, left_suffix, right_suffix, left_name, right_name, improvement_label):
    rows = []
    for product, g in matched_df.groupby("product"):
        left_mention_pos = _safe_mean(g[f"average_mention_position_proxy{left_suffix}"])
        right_mention_pos = _safe_mean(g[f"average_mention_position_proxy{right_suffix}"])
        left_citation_pos = _safe_mean(g[f"citation_position_proxy{left_suffix}"])
        right_citation_pos = _safe_mean(g[f"citation_position_proxy{right_suffix}"])

        rows.append({
            "product": product,
            f"{left_name}_brand_mention_rate": _safe_mean(g[f"brand_mentioned{left_suffix}"]),
            f"{right_name}_brand_mention_rate": _safe_mean(g[f"brand_mentioned{right_suffix}"]),
            "delta_brand_mention_rate": _safe_mean(g[f"brand_mentioned{right_suffix}"] - g[f"brand_mentioned{left_suffix}"]),

            f"{left_name}_mention_share_of_voice": _safe_mean(g[f"mention_share_of_voice{left_suffix}"]),
            f"{right_name}_mention_share_of_voice": _safe_mean(g[f"mention_share_of_voice{right_suffix}"]),
            "delta_mention_share_of_voice": _safe_mean(g[f"mention_share_of_voice{right_suffix}"] - g[f"mention_share_of_voice{left_suffix}"]),

            f"{left_name}_citation_share_of_voice": _safe_mean(g[f"citation_share_of_voice{left_suffix}"]),
            f"{right_name}_citation_share_of_voice": _safe_mean(g[f"citation_share_of_voice{right_suffix}"]),
            "delta_citation_share_of_voice": _safe_mean(g[f"citation_share_of_voice{right_suffix}"] - g[f"citation_share_of_voice{left_suffix}"]),

            f"{left_name}_average_mention_position": left_mention_pos,
            f"{right_name}_average_mention_position": right_mention_pos,
            f"mention_position_improvement_{improvement_label}": left_mention_pos - right_mention_pos,

            f"{left_name}_average_citation_position": left_citation_pos,
            f"{right_name}_average_citation_position": right_citation_pos,
            f"citation_position_improvement_{improvement_label}": left_citation_pos - right_citation_pos,

            f"{left_name}_first_mention_score": _safe_mean(g[f"first_mention_score{left_suffix}"]),
            f"{right_name}_first_mention_score": _safe_mean(g[f"first_mention_score{right_suffix}"]),
            "delta_first_mention_score": _safe_mean(g[f"first_mention_score{right_suffix}"] - g[f"first_mention_score{left_suffix}"]),

            f"{left_name}_brand_context_sentiment_score": _safe_mean(g[f"brand_context_sentiment_score{left_suffix}"]),
            f"{right_name}_brand_context_sentiment_score": _safe_mean(g[f"brand_context_sentiment_score{right_suffix}"]),
            "delta_brand_context_sentiment_score": _safe_mean(g[f"brand_context_sentiment_score{right_suffix}"] - g[f"brand_context_sentiment_score{left_suffix}"]),

            "source_domain_share_status": "not_available_controlled_product_label_citations_only",
        })
    out = pd.DataFrame(rows)
    out["provider_style_visibility_delta"] = (
        0.20 * out["delta_brand_mention_rate"]
        + 0.25 * out["delta_mention_share_of_voice"]
        + 0.25 * out["delta_citation_share_of_voice"]
        + 0.15 * (out[f"mention_position_improvement_{improvement_label}"] / N_PRODUCTS)
        + 0.15 * (out[f"citation_position_improvement_{improvement_label}"] / N_PRODUCTS)
    )
    return out

provider_answer_metric_delta_by_product = _provider_delta_rows(
    matched_complete,
    left_suffix="_current",
    right_suffix="_geo",
    left_name="current",
    right_name="geo",
    improvement_label="geo_minus_current",
)
geo_vs_current_delta_by_product = geo_vs_current_delta_by_product.merge(
    provider_answer_metric_delta_by_product,
    on="product",
    how="left",
)
geo_vs_current_delta_by_product["replacement_visibility_delta"] = geo_vs_current_delta_by_product["geo_vs_current_advantage_score"]

overall_geo_vs_current_summary["provider_style_metric_note"] = PROVIDER_STYLE_METRIC_NOTE
for _col in [
    "delta_brand_mention_rate",
    "delta_mention_share_of_voice",
    "delta_citation_share_of_voice",
    "provider_style_visibility_delta",
    "replacement_visibility_delta",
    "delta_brand_context_sentiment_score",
]:
    overall_geo_vs_current_summary[f"mean_{_col}"] = geo_vs_current_delta_by_product[_col].mean()

display(geo_vs_current_delta_by_product[[
    "product",
    "delta_brand_mention_rate",
    "delta_mention_share_of_voice",
    "delta_citation_share_of_voice",
    "mention_position_improvement_geo_minus_current",
    "citation_position_improvement_geo_minus_current",
    "delta_brand_context_sentiment_score",
    "provider_style_visibility_delta",
    "replacement_visibility_delta",
]])


,product,delta_brand_mention_rate,delta_mention_share_of_voice,delta_citation_share_of_voice,mention_position_improvement_geo_minus_current,citation_position_improvement_geo_minus_current,delta_brand_context_sentiment_score,provider_style_visibility_delta,replacement_visibility_delta
0,Away,0.00,0.000476,0.007632,0.600000,0.70,-0.111385,0.041027,0.103930
1,Rimowa,0.00,-0.036217,-0.004679,0.733333,1.00,0.180144,0.041776,0.068070
2,July,0.00,-0.021081,-0.011155,0.050000,-0.05,-0.070269,-0.008059,0.008549
3,BR,-0.05,-0.057580,-0.052015,-0.200000,-0.10,-0.053312,-0.046399,-0.040820
4,Century,0.00,0.056416,-0.043964,0.250000,-0.45,0.029545,-0.002887,-0.085725


## Save CSV outputs


In [13]:
file_manifest_comparison03.to_csv(OUTPUT_DIR / "file_manifest_comparison03.csv", index=False, encoding="utf-8-sig")
baseline_answers.to_csv(OUTPUT_DIR / "current_baseline_parsed_answers.csv", index=False, encoding="utf-8-sig")
geo_answers.to_csv(OUTPUT_DIR / "geo_replacement_parsed_answers_by_file.csv", index=False, encoding="utf-8-sig")
coverage_by_file.to_csv(OUTPUT_DIR / "question_coverage_by_file.csv", index=False, encoding="utf-8-sig")
question_coverage_summary.to_csv(OUTPUT_DIR / "question_coverage_summary.csv", index=False, encoding="utf-8-sig")
geo_prompt_chunk_coverage_by_product.to_csv(OUTPUT_DIR / "geo_replacement_prompt_chunk_coverage_by_product.csv", index=False, encoding="utf-8-sig")
matching_coverage_summary.to_csv(OUTPUT_DIR / "matching_coverage_summary.csv", index=False, encoding="utf-8-sig")
missing_current_matches.to_csv(OUTPUT_DIR / "missing_current_baseline_matches.csv", index=False, encoding="utf-8-sig")
baseline_qp_metrics.to_csv(OUTPUT_DIR / "current_baseline_question_product_metrics.csv", index=False, encoding="utf-8-sig")
geo_qp_metrics.to_csv(OUTPUT_DIR / "geo_replacement_question_product_metrics.csv", index=False, encoding="utf-8-sig")
matched_complete.to_csv(OUTPUT_DIR / "matched_geo_current_question_product_delta.csv", index=False, encoding="utf-8-sig")
current_product_summary_matched.to_csv(OUTPUT_DIR / "current_product_summary_matched.csv", index=False, encoding="utf-8-sig")
geo_product_summary_matched.to_csv(OUTPUT_DIR / "geo_product_summary_matched.csv", index=False, encoding="utf-8-sig")
geo_vs_current_delta_by_product.to_csv(OUTPUT_DIR / "geo_vs_current_delta_by_product.csv", index=False, encoding="utf-8-sig")
provider_answer_metric_delta_by_product.to_csv(OUTPUT_DIR / "provider_answer_metric_delta_by_product.csv", index=False, encoding="utf-8-sig")
overall_geo_vs_current_summary.to_csv(OUTPUT_DIR / "overall_geo_vs_current_summary.csv", index=False, encoding="utf-8-sig")
product_text_load_status.to_csv(OUTPUT_DIR / "product_text_load_status.csv", index=False, encoding="utf-8-sig")

print("Saved CSV files to:")
print(OUTPUT_DIR)
for p in sorted(OUTPUT_DIR.glob("*.csv")):
    print("-", p.name)


Saved CSV files to:
C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\supplementary_check\results\tables\03_2026_current_vs_2023_2024_geo_rewrite_baseline_current_chunked_based
- current_baseline_parsed_answers.csv
- current_baseline_question_product_metrics.csv
- current_product_summary_matched.csv
- file_manifest_comparison03.csv
- geo_product_summary_matched.csv
- geo_replacement_parsed_answers_by_file.csv
- geo_replacement_prompt_chunk_coverage_by_product.csv
- geo_replacement_question_product_metrics.csv
- geo_vs_current_delta_by_product.csv
- matched_geo_current_question_product_delta.csv
- matching_coverage_summary.csv
- missing_current_baseline_matches.csv
- overall_geo_vs_current_summary.csv
- product_text_load_status.csv
- provider_answer_metric_delta_by_product.csv
- question_coverage_by_file.csv
- question_coverage_summary.csv


## Quick interpretation table


In [14]:

cols = [
    "product",
    "direction_label",
    "geo_vs_current_advantage_score",
    "provider_style_visibility_delta",
    "replacement_visibility_delta",
    "delta_rank_score_proxy",
    "delta_citation_rate",
    "delta_brand_mention_rate",
    "delta_citation_share_of_voice",
    "delta_mention_share_of_voice",
    "mention_position_improvement_geo_minus_current",
    "citation_position_improvement_geo_minus_current",
    "delta_first_citation_score",
    "delta_top1_cited_share",
    "delta_feature_coverage",
    "delta_pais",
    "matched_n_question_contexts",
    "question_ids",
    "prompt_sets",
]
cols = [c for c in cols if c in geo_vs_current_delta_by_product.columns]
print("Quick check: provider-style answer metrics are now included alongside the original GEO-over-current advantage metrics.")
print("source/domain share is not computed because these runs use controlled product-label citations, not external URLs/domains.")
display(geo_vs_current_delta_by_product[cols])


Quick check: provider-style answer metrics are now included alongside the original GEO-over-current advantage metrics.
source/domain share is not computed because these runs use controlled product-label citations, not external URLs/domains.


,product,direction_label,geo_vs_current_advantage_score,provider_style_visibility_delta,replacement_visibility_delta,delta_rank_score_proxy,delta_citation_rate,delta_brand_mention_rate,delta_citation_share_of_voice,delta_mention_share_of_voice,mention_position_improvement_geo_minus_current,citation_position_improvement_geo_minus_current,delta_first_citation_score,delta_top1_cited_share,delta_feature_coverage,delta_pais,matched_n_question_contexts,question_ids,prompt_sets
0,Away,geo_rewrite_stronger_than_current,0.103930,0.041027,0.103930,0.130000,0.05,0.00,0.007632,0.000476,0.600000,0.70,0.140000,0.150000,0.078947,0.030349,20,"1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20",1-5;6-10;11-15;16-20
1,Rimowa,geo_rewrite_stronger_than_current,0.068070,0.041776,0.068070,0.146667,0.00,0.00,-0.004679,-0.036217,0.733333,1.00,0.083704,0.066667,0.045614,0.001016,15,"6,7,8,9,10,11,12,13,14,15,16,17,18,19,20",6-10;11-15;16-20
2,July,similar_or_small_change,0.008549,-0.008059,0.008549,0.010000,0.00,0.00,-0.011155,-0.021081,0.050000,-0.05,0.008333,0.050000,-0.015789,-0.015384,20,"1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20",1-5;6-10;11-15;16-20
3,BR,similar_or_small_change,-0.040820,-0.046399,-0.040820,-0.040000,-0.05,-0.05,-0.052015,-0.057580,-0.200000,-0.10,-0.049167,-0.050000,0.023684,-0.058550,20,"1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20",1-5;6-10;11-15;16-20
4,Century,current_2026_stronger_than_geo_rewrite,-0.085725,-0.002887,-0.085725,-0.090000,-0.25,0.00,-0.043964,0.056416,0.250000,-0.45,-0.036111,0.050000,-0.071053,-0.063978,20,"1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20",1-5;6-10;11-15;16-20


In [15]:
display(file_manifest_comparison03.sort_values(["included", "file_role", "filename"], ascending=[False, True, True]))


,file_role,path,filename,detected_product,prompt_start_from_filename,included,reason
0,baseline_current_included,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\supplementary_check\results\shopping_answers\baseline_curr\baseline_1.txt,baseline_1.txt,,1,True,all-current 2026 baseline answer file from baseline_*_curr chunk
2,baseline_current_included,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\supplementary_check\results\shopping_answers\baseline_curr\baseline_11.txt,baseline_11.txt,,11,True,all-current 2026 baseline answer file from baseline_*_curr chunk
3,baseline_current_included,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\supplementary_check\results\shopping_answers\baseline_curr\baseline_16.txt,baseline_16.txt,,16,True,all-current 2026 baseline answer file from baseline_*_curr chunk
1,baseline_current_included,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\supplementary_check\results\shopping_answers\baseline_curr\baseline_6.txt,baseline_6.txt,,6,True,all-current 2026 baseline answer file from baseline_*_curr chunk
4,comparison03_geo_rewrite_replacement_included,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\supplementary_check\results\shopping_answers\03_2026_current_vs_2023_2024_geo_rewrite\away_1.txt,away_1.txt,Away,1,True,one-product GEO rewrite replacement file against all-current context
5,comparison03_geo_rewrite_replacement_included,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\supplementary_check\results\shopping_answers\03_2026_current_vs_2023_2024_geo_rewrite\away_11.txt,away_11.txt,Away,11,True,one-product GEO rewrite replacement file against all-current context
6,comparison03_geo_rewrite_replacement_included,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\supplementary_check\results\shopping_answers\03_2026_current_vs_2023_2024_geo_rewrite\away_16.txt,away_16.txt,Away,16,True,one-product GEO rewrite replacement file against all-current context
7,comparison03_geo_rewrite_replacement_included,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\supplementary_check\results\shopping_answers\03_2026_current_vs_2023_2024_geo_rewrite\away_6.txt,away_6.txt,Away,6,True,one-product GEO rewrite replacement file against all-current context
8,comparison03_geo_rewrite_replacement_included,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\supplementary_check\results\shopping_answers\03_2026_current_vs_2023_2024_geo_rewrite\br_1.txt,br_1.txt,BR,1,True,one-product GEO rewrite replacement file against all-current context
9,comparison03_geo_rewrite_replacement_included,C:\Users\junse\Documents\research\Finance Research\GEO_carry-on\supplementary_check\results\shopping_answers\03_2026_current_vs_2023_2024_geo_rewrite\br_11.txt,br_11.txt,BR,11,True,one-product GEO rewrite replacement file against all-current context
